In [ ]:
# =========================
# STEP 1: Mount Drive, set project paths, clone/sync GitHub repo
# =========================

from google.colab import drive
from pathlib import Path
import os, subprocess, textwrap, json, datetime, sys

# 1. Mount Google Drive
drive.mount("/content/drive")

# 2. Define project root
PROJECT_ROOT = Path("/content/drive/MyDrive/MCI_Project")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# 3. Define GitHub repo details
GITHUB_REPO_URL = "https://github.com/SANGHATI23/mci-cardiomyopathy-concordance.git"
REPO_DIR = PROJECT_ROOT / "mci-cardiomyopathy-concordance"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPO_DIR:", REPO_DIR)

# 4. Helper function to run shell commands safely
def run_cmd(cmd, cwd=None, check=True):
    print("\n$", cmd)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")
    return result

# 5. Clone repo if missing, otherwise update it
if not REPO_DIR.exists():
    run_cmd(f"git clone {GITHUB_REPO_URL} {REPO_DIR}")
else:
    print("Repo already exists. Checking status and pulling latest main...")
    run_cmd("git status", cwd=REPO_DIR, check=False)
    run_cmd("git remote -v", cwd=REPO_DIR, check=False)
    run_cmd("git fetch origin", cwd=REPO_DIR)
    run_cmd("git checkout main", cwd=REPO_DIR, check=False)
    run_cmd("git pull origin main", cwd=REPO_DIR, check=False)

# 6. Create a safe working branch for resource-paper reframing
branch_name = "resource-paper-reframe"

# Check if branch exists locally
branches = run_cmd("git branch --list", cwd=REPO_DIR, check=False).stdout

if branch_name in branches:
    run_cmd(f"git checkout {branch_name}", cwd=REPO_DIR)
else:
    run_cmd(f"git checkout -b {branch_name}", cwd=REPO_DIR)

# 7. Create standard output folders for the new resource-paper direction
folders = [
    "manuscript/resource_paper",
    "docs/resource_framing",
    "results/resource_tables",
    "results/figures/resource_paper",
    "scripts/resource_paper",
    "scripts/scrna_contextualization",
    "shiny_app",
    "metadata"
]

for folder in folders:
    path = REPO_DIR / folder
    path.mkdir(parents=True, exist_ok=True)
    print("Created/confirmed:", path)

# 8. Create run metadata
metadata = {
    "step": "STEP_1_REPO_SETUP",
    "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "project_root": str(PROJECT_ROOT),
    "repo_dir": str(REPO_DIR),
    "github_repo": GITHUB_REPO_URL,
    "branch": branch_name,
    "goal": "Reframe MCI project as a database/resource manuscript rather than a high-impact discovery manuscript."
}

metadata_path = REPO_DIR / "metadata" / "resource_reframe_notebook_run_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2))

print("\nSaved metadata:", metadata_path)

# 9. Show repo tree top level
print("\nTop-level repo contents:")
for item in sorted(REPO_DIR.iterdir()):
    print(" -", item.name)

print("\nSTEP 1 COMPLETE.")
print("Next: paste the output here, especially any git errors.")

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/MCI_Project
REPO_DIR: /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance

$ git clone https://github.com/SANGHATI23/mci-cardiomyopathy-concordance.git /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance
Cloning into '/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance'...
Updating files:   7% (22/297)
Updating files:   8% (24/297)
Updating files:   9% (27/297)
Updating files:  10% (30/297)
Updating files:  11% (33/297)
Updating files:  12% (36/297)
Updating files:  13% (39/297)
Updating files:  14% (42/297)
Updating files:  15% (45/297)
Updating files:  16% (48/297)
Updating files:  16% (50/297)
Updating files:  17% (51/297)
Updating files:  18% (54/297)
Updating files:  19% (57/297)
Updating files:  20% (60/297)
Updating files:  21% (63/297)
Updating files:  21% (64/297)
Updating files:  22% (66/297)
Updating files:  23% (69/297)
Updating files:  24% (72/297)
Updating files:  25% 

/tmp/ipykernel_2654/3034754927.py:81: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",


In [ ]:
# =========================
# STEP 2: Create resource-paper framing documents
# =========================

from pathlib import Path
import json, datetime, textwrap

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")

assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

resource_dir = REPO_DIR / "manuscript" / "resource_paper"
docs_dir = REPO_DIR / "docs" / "resource_framing"
metadata_dir = REPO_DIR / "metadata"

resource_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

# -------------------------
# 1. Resource-paper title and positioning
# -------------------------

title_positioning = """# Resource-Paper Reframing for the MCI Cardiomyopathy Project

## Proposed resource-paper title

Molecular Concordance Index: a reproducible transcriptomic resource for evaluating ClinVar-annotated cardiomyopathy genes across human HCM and DCM cohorts

## Short title

MCI cardiomyopathy concordance resource

## Core reframing

This manuscript should not be positioned as a high-impact discovery article whose success depends on statistically significant biological hypotheses. The current results are honest, useful, and reproducible, but the main hypothesis tests are directionally supportive rather than statistically significant.

The stronger framing is a database/resource and computational framework paper.

The central contribution is:

A reusable Molecular Concordance Index resource that evaluates whether ClinVar-annotated cardiomyopathy genes show reproducible transcript-level evidence across public human heart datasets, while integrating bootstrap uncertainty, GTEx left-ventricle baseline benchmarking, DCM generalization, held-out validation, GWAS convergence analysis, and downloadable/browser-ready outputs.

## Main resource claim

Clinical pathogenicity annotation, cross-cohort transcriptomic reproducibility, normal-tissue variability, and genetic association are related but non-equivalent evidence layers. The MCI resource makes these layers visible gene by gene.

## What the paper should claim

1. We constructed a reproducible gene-level concordance resource for ClinVar-annotated HCM and DCM genes.
2. We operationalized transcriptomic concordance using direction agreement, effect-size consistency, and statistical reproducibility.
3. We attached bootstrap confidence intervals and tier-stability estimates to every eligible gene.
4. We benchmarked disease-associated variability against GTEx v8 Heart - Left Ventricle baseline variability.
5. We tested DCM generalization, held-out replication, and GWAS convergence as validation modules.
6. We provide browser-ready and downloadable tables to support cardiomyopathy target-evidence review.

## What the paper should not claim

1. It should not claim that sarcomeric and non-sarcomeric mechanisms are definitively separated by MCI.
2. It should not claim that DCM generalization confirmed the expected mechanism pattern.
3. It should not claim that held-out validation or GWAS convergence reached statistical significance.
4. It should not claim direct wet-lab target validation.
5. It should not claim clinical actionability without orthogonal validation.

## Best-fit manuscript type

Database/resource paper, software/resource paper, or methods-resource paper.

Potential targets:
- NAR Genomics and Bioinformatics
- Bioinformatics Advances
- Database: The Journal of Biological Databases and Curation
- GigaScience
- Scientific Data, if data packaging and metadata become the primary focus
"""

(resource_dir / "00_RESOURCE_PAPER_POSITIONING.md").write_text(title_positioning)


# -------------------------
# 2. Revised abstract
# -------------------------

abstract = """# Revised Resource-Paper Abstract

Clinical pathogenicity annotation is essential for cardiomyopathy genetics, but it does not establish whether a gene shows reproducible transcript-level perturbation across independent human disease cohorts. This creates a translational informatics gap: genes may be clinically important while remaining unstable, cohort-specific, or indistinguishable from normal-tissue variability at the bulk-transcriptome level. We present the Molecular Concordance Index (MCI), a reproducible resource and scoring framework for evaluating transcriptomic concordance of ClinVar-annotated hypertrophic cardiomyopathy (HCM) and dilated cardiomyopathy (DCM) genes across public human heart datasets.

MCI integrates three evidence layers: direction agreement across cohorts, effect-size consistency, and statistical reproducibility. We applied the framework to public HCM cohorts, extended the analysis to DCM generalization datasets, added 1,000-iteration cohort-level bootstrap confidence intervals, and benchmarked disease-associated variability against GTEx v8 Heart - Left Ventricle expression variability. The resulting resource assigns each gene a concordance score, bootstrap interval, tier classification, GTEx low-confidence flag, and per-cohort differential-expression evidence.

Across 49 HCM genes, the resource identified heterogeneous transcript-level behavior: 18 genes were classified as high concordance, 8 as moderate, 21 as unstable, and 2 as insufficient coverage in the primary HCM analysis. Bootstrap tier-majority assignments were largely stable, but confidence intervals were wide because only three primary HCM cohorts were available. GTEx benchmarking showed that most genes had disease-associated variability that did not exceed normal left-ventricle baseline variability, indicating that transcript-concordant genes should not automatically be interpreted as unqualified transcript-level targets. Pre-specified mechanism-stratified, DCM generalization, held-out replication, and GWAS convergence analyses were directionally informative but did not reach statistical significance.

The MCI resource therefore provides a transparent evidence-auditing layer for cardiomyopathy target evaluation rather than a binary discovery test. It enables researchers to distinguish clinically annotated genes with reproducible transcriptomic support from genes whose disease signal is unstable, baseline-confounded, disease-context-specific, or underpowered. All resource tables, figures, and browser-ready outputs are designed for public reuse and extension, including future single-cell and genotype-stratified modules.
"""

(resource_dir / "01_REVISED_RESOURCE_ABSTRACT.md").write_text(abstract)


# -------------------------
# 3. Revised contribution bullets
# -------------------------

contributions = """# Revised Contributions for Resource-Paper Manuscript

The manuscript should present the contribution as a reusable evidence resource, not as a failed hypothesis-driven discovery study.

## Contribution 1: MCI scoring framework

We define and implement a reproducible Molecular Concordance Index that summarizes transcriptomic reproducibility of ClinVar-annotated cardiomyopathy genes across independent human heart cohorts using direction agreement, effect-size consistency, and statistical reproducibility.

## Contribution 2: Cardiomyopathy concordance table

We generate a gene-level HCM/DCM concordance resource containing primary MCI, adjusted MCI, tier labels, bootstrap confidence intervals, bootstrap tier probabilities, GTEx low-confidence status, and per-cohort differential-expression evidence.

## Contribution 3: Baseline-context interpretation

We benchmark disease-associated transcript variability against GTEx v8 Heart - Left Ventricle expression variability, separating cross-cohort concordance from variability that may fall within normal human left-ventricle expression range.

## Contribution 4: Validation modules

We report DCM generalization, held-out validation, and GWAS convergence modules as transparent evidence layers. These modules are not overclaimed as statistically definitive. Their value is to show where transcriptomic concordance does or does not align with independent disease context, replication, and genetic association evidence.

## Contribution 5: Resource usability

We organize outputs as downloadable and browser-ready tables so that researchers can query genes, inspect per-cohort evidence, view uncertainty, and decide whether transcript-level evidence is strong enough to support downstream target-validation work.

## Contribution 6: Extension path

We define a clear extension path for single-cell RNA-seq contextualization, genotype-stratified scoring, proteomic validation, and larger biobank-scale replication.
"""

(resource_dir / "02_REVISED_CONTRIBUTIONS.md").write_text(contributions)


# -------------------------
# 4. Claim boundary statement
# -------------------------

claim_boundary = """# Submission-Ready Claim Boundary Statement

This study presents MCI as a reproducible transcriptomic concordance resource for ClinVar-annotated cardiomyopathy genes. The resource is intended to support evidence auditing, target prioritization, and hypothesis generation.

The study does not claim that transcriptomic concordance proves disease causality, clinical actionability, or therapeutic tractability. It also does not claim that the mechanism-stratified HCM analysis, DCM generalization analysis, held-out validation, or GWAS convergence analysis reached definitive statistical significance.

The major finding is that cardiomyopathy genes differ substantially in their cross-cohort transcript-level reproducibility and in whether their disease-associated variability exceeds GTEx normal left-ventricle baseline variability. This distinction is valuable because it shows that clinical pathogenicity annotation, transcriptomic reproducibility, normal-tissue variability, and genetic association are non-equivalent evidence layers.

Therefore, MCI should be interpreted as a transparent evidence-auditing resource and translational gate, not as a standalone clinical decision tool.
"""

(docs_dir / "CLAIM_BOUNDARY_STATEMENT.md").write_text(claim_boundary)


# -------------------------
# 5. Manuscript outline
# -------------------------

outline = """# Resource-Paper Manuscript Outline

## Title

Molecular Concordance Index: a reproducible transcriptomic resource for evaluating ClinVar-annotated cardiomyopathy genes across human HCM and DCM cohorts

## Abstract

Use the revised resource-paper abstract in `01_REVISED_RESOURCE_ABSTRACT.md`.

## Introduction

1. Cardiomyopathy genetics depends heavily on ClinVar and disease-gene annotation.
2. Clinical pathogenicity does not guarantee reproducible transcriptomic perturbation.
3. Public bulk transcriptomic cohorts provide an opportunity to audit transcript-level reproducibility.
4. Current gaps:
   - no reusable gene-level concordance resource for ClinVar cardiomyopathy genes,
   - limited separation of disease-cohort reproducibility from normal-tissue variability,
   - limited transparent reporting of underpowered or non-significant validation attempts,
   - limited browser-ready tools for gene-level evidence review.
5. This study presents MCI as a reproducible resource.

## Results

### 1. Resource construction and cohort coverage
Report HCM and DCM datasets, ClinVar gene universe, and score-eligible genes.

### 2. Primary HCM MCI resource
Report 49 HCM genes scored, tier counts, and top high/moderate/unstable genes.

### 3. Bootstrap uncertainty and tier stability
Report 1,000-iteration bootstrap, 47 genes with CIs, wide intervals, and tier-majority behavior.

### 4. GTEx baseline benchmarking
Report GTEx v8 Heart - Left Ventricle sample extraction, 48 of 49 GTEx matches, and low-confidence flags.

### 5. Mechanism-stratified use case
Report sarcomeric vs non-sarcomeric comparison as a pre-specified demonstration analysis, not as the main discovery claim.

### 6. DCM generalization module
Report DCM results as disease-context evaluation. Emphasize that the expected pattern was not confirmed.

### 7. Held-out and GWAS validation modules
Report directionally supportive but non-significant held-out and GWAS results.

### 8. Browser-ready resource outputs
Describe downloadable tables, figures, and planned or implemented Shiny browser.

### 9. Optional scRNA-seq contextualization module
If completed, add cell-type enrichment/expression context for MCI genes.

## Methods

1. Dataset acquisition and eligibility
2. ClinVar HCM/DCM gene universe construction
3. Differential-expression processing
4. Harmonization and batch-aware processing
5. MCI formula
6. Tier assignment
7. Bootstrap confidence intervals
8. GTEx baseline adjustment
9. DCM generalization
10. Held-out validation
11. GWAS convergence
12. Shiny/browser implementation
13. Optional scRNA-seq contextualization

## Discussion

1. MCI as a resource for translational evidence auditing.
2. Why clinical pathogenicity and transcript reproducibility are non-equivalent.
3. How GTEx benchmarking changes interpretation of high-MCI genes.
4. What non-significant validation modules mean in a resource-paper context.
5. Use cases for target selection, biomarker review, and study design.
6. Limitations:
   - few cohorts,
   - bulk tissue only,
   - missing genotype resolution,
   - no wet-lab validation,
   - incomplete scRNA/proteomic context.
7. Future directions:
   - scRNA-seq,
   - genotype-stratified MCI,
   - proteomics,
   - larger biobank-scale validation,
   - browser expansion.

## Data and Code Availability

Include GitHub, Zenodo, Shiny URL, and versioned CSV table locations once finalized.
"""

(resource_dir / "03_RESOURCE_MANUSCRIPT_OUTLINE.md").write_text(outline)


# -------------------------
# 6. README for resource-paper work
# -------------------------

readme = """# MCI Resource-Paper Reframe

This folder contains the resource-paper reframing files for the Molecular Concordance Index cardiomyopathy project.

## Why this reframe is needed

The current empirical results are scientifically useful but not strong enough to support a high-impact biological discovery claim because the main mechanism, DCM generalization, held-out validation, and GWAS convergence tests are directionally informative but non-significant.

The stronger and more publishable framing is a database/resource paper.

## Main framing

MCI is a reproducible transcriptomic evidence-auditing resource for ClinVar-annotated cardiomyopathy genes.

## Key files

- `00_RESOURCE_PAPER_POSITIONING.md`: overall positioning and claim boundaries.
- `01_REVISED_RESOURCE_ABSTRACT.md`: resource-paper abstract.
- `02_REVISED_CONTRIBUTIONS.md`: contribution list.
- `03_RESOURCE_MANUSCRIPT_OUTLINE.md`: new manuscript structure.

## Next steps

1. Build final resource table.
2. Build or update Shiny browser.
3. Add single-cell contextualization module if feasible.
4. Rewrite manuscript using this structure.
5. Push resource outputs to GitHub.
6. Deposit stable data release to Zenodo.
"""

(resource_dir / "README.md").write_text(readme)


# -------------------------
# 7. Metadata
# -------------------------

metadata = {
    "step": "STEP_2_RESOURCE_FRAMING_FILES",
    "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "files_created": [
        str(resource_dir / "00_RESOURCE_PAPER_POSITIONING.md"),
        str(resource_dir / "01_REVISED_RESOURCE_ABSTRACT.md"),
        str(resource_dir / "02_REVISED_CONTRIBUTIONS.md"),
        str(resource_dir / "03_RESOURCE_MANUSCRIPT_OUTLINE.md"),
        str(resource_dir / "README.md"),
        str(docs_dir / "CLAIM_BOUNDARY_STATEMENT.md"),
    ],
    "purpose": "Reframe the MCI cardiomyopathy project as a resource/database paper rather than a discovery article."
}

(metadata_dir / "step2_resource_framing_metadata.json").write_text(json.dumps(metadata, indent=2))

# -------------------------
# 8. Show created files
# -------------------------

print("STEP 2 COMPLETE.")
print("\nCreated files:")
for file_path in metadata["files_created"]:
    print(" -", file_path)

print("\nPreview of revised abstract:\n")
print(abstract[:1200])

STEP 2 COMPLETE.

Created files:
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/manuscript/resource_paper/00_RESOURCE_PAPER_POSITIONING.md
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/manuscript/resource_paper/01_REVISED_RESOURCE_ABSTRACT.md
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/manuscript/resource_paper/02_REVISED_CONTRIBUTIONS.md
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/manuscript/resource_paper/03_RESOURCE_MANUSCRIPT_OUTLINE.md
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/manuscript/resource_paper/README.md
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/docs/resource_framing/CLAIM_BOUNDARY_STATEMENT.md

Preview of revised abstract:

# Revised Resource-Paper Abstract

Clinical pathogenicity annotation is essential for cardiomyopathy genetics, but it does not establish whether a gene shows reproducible transcript-level perturbation acro

/tmp/ipykernel_2654/2991380731.py:295: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",


In [ ]:
# =========================
# STEP 3: Build database/resource-ready master table
# =========================

from pathlib import Path
import pandas as pd
import numpy as np
import json, datetime, re

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")
assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

resource_table_dir = REPO_DIR / "results" / "resource_tables"
qc_dir = REPO_DIR / "results" / "quality_control" / "resource_tables"
resource_table_dir.mkdir(parents=True, exist_ok=True)
qc_dir.mkdir(parents=True, exist_ok=True)

# -------------------------
# Helper functions
# -------------------------

def find_files(patterns, root=REPO_DIR):
    hits = []
    for pattern in patterns:
        hits.extend(list(root.rglob(pattern)))
    # remove duplicates, sort by modified time newest first
    hits = sorted(set(hits), key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)
    return hits

def read_csv_safe(path):
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"Could not read {path}: {e}")
        return None

def standardize_gene_symbol(series):
    return (
        series.astype(str)
        .str.strip()
        .str.upper()
        .replace({"NAN": np.nan, "NONE": np.nan, "": np.nan})
    )

def pick_first_existing(patterns, label):
    hits = find_files(patterns)
    print(f"\n{label}: found {len(hits)} candidate files")
    for i, h in enumerate(hits[:10], start=1):
        print(f"  {i}. {h.relative_to(REPO_DIR)}")
    if hits:
        print(f"Selected for {label}: {hits[0].relative_to(REPO_DIR)}")
        return hits[0]
    print(f"WARNING: No file found for {label}")
    return None

# -------------------------
# 1. Locate latest core tables
# -------------------------

hcm_final_path = pick_first_existing(
    [
        "TASK6_FINAL_HCM_MCI_WITH_GTEx_AND_BOOTSTRAP_CI.csv",
        "*HCM*MCI*GTEx*BOOTSTRAP*.csv",
        "*FINAL*HCM*MCI*SCORE*BOOTSTRAP*.csv",
        "*HCM_MCI*bootstrap*.csv",
    ],
    "Final HCM MCI + GTEx + bootstrap table"
)

dcm_paths = find_files([
    "*DCM*MCI*.csv",
    "*TASK8*DCM*.csv",
    "*DCM_generalization*.csv",
])
print(f"\nDCM candidate tables found: {len(dcm_paths)}")
for i, h in enumerate(dcm_paths[:15], start=1):
    print(f"  {i}. {h.relative_to(REPO_DIR)}")

validation_paths = find_files([
    "*TASK9*.csv",
    "*held*out*.csv",
    "*GWAS*.csv",
    "*validation*.csv",
])
print(f"\nValidation candidate tables found: {len(validation_paths)}")
for i, h in enumerate(validation_paths[:20], start=1):
    print(f"  {i}. {h.relative_to(REPO_DIR)}")

# -------------------------
# 2. Load HCM table
# -------------------------

if hcm_final_path is None:
    raise FileNotFoundError(
        "Could not find final HCM MCI table. Make sure Task 6 outputs exist in the repo."
    )

hcm = read_csv_safe(hcm_final_path)
if hcm is None:
    raise RuntimeError("Failed to load HCM table.")

print("\nLoaded HCM table:", hcm.shape)
print("HCM columns:", list(hcm.columns))

# -------------------------
# 3. Standardize HCM table into resource schema
# -------------------------

# Detect likely columns
gene_col = "gene_symbol" if "gene_symbol" in hcm.columns else None
if gene_col is None:
    possible = [c for c in hcm.columns if "gene" in c.lower() and "symbol" in c.lower()]
    if possible:
        gene_col = possible[0]
    else:
        raise ValueError("No gene_symbol column detected in HCM table.")

def get_col(df, candidates, default=np.nan):
    for c in candidates:
        if c in df.columns:
            return df[c]
    return default

resource = pd.DataFrame()
resource["gene_symbol"] = standardize_gene_symbol(hcm[gene_col])
resource["disease_group"] = "HCM"
resource["source_resource_layer"] = "primary_HCM_bulk_RNA_resource"
resource["stratum"] = get_col(hcm, ["stratum", "mechanism_group", "gene_stratum"])
resource["is_prespecified_h4_gene"] = get_col(hcm, ["is_prespecified_h4_gene"], False)

resource["MCI"] = pd.to_numeric(get_col(hcm, ["MCI", "MCI_original_from_step5", "point_MCI"]), errors="coerce")
resource["Adj_MCI"] = pd.to_numeric(get_col(hcm, ["Adj_MCI", "GTEx_adjusted_MCI", "adjusted_MCI"]), errors="coerce")

resource["MCI_tier"] = get_col(hcm, ["MCI_tier", "tier", "primary_MCI_tier"])
resource["bootstrap_majority_tier"] = get_col(hcm, ["bootstrap_majority_tier", "majority_tier"])
resource["MCI_CI95_lower"] = pd.to_numeric(get_col(hcm, ["MCI_CI95_lower", "CI_lower", "bootstrap_CI_lower"]), errors="coerce")
resource["MCI_CI95_upper"] = pd.to_numeric(get_col(hcm, ["MCI_CI95_upper", "CI_upper", "bootstrap_CI_upper"]), errors="coerce")
resource["bootstrap_median_MCI"] = pd.to_numeric(get_col(hcm, ["bootstrap_median_MCI", "median_MCI"]), errors="coerce")
resource["bootstrap_prob_HIGH"] = pd.to_numeric(get_col(hcm, ["bootstrap_prob_HIGH", "P_HIGH"]), errors="coerce")
resource["bootstrap_prob_MODERATE"] = pd.to_numeric(get_col(hcm, ["bootstrap_prob_MODERATE", "P_MODERATE"]), errors="coerce")
resource["bootstrap_prob_UNSTABLE"] = pd.to_numeric(get_col(hcm, ["bootstrap_prob_UNSTABLE", "P_UNSTABLE"]), errors="coerce")

resource["sigma_disease"] = pd.to_numeric(get_col(hcm, ["sigma_disease"]), errors="coerce")
resource["sigma_GTEx"] = pd.to_numeric(get_col(hcm, ["sigma_GTEx"]), errors="coerce")
resource["sigma_disease_to_GTEx_ratio"] = pd.to_numeric(
    get_col(hcm, ["sigma_disease_to_GTEx_ratio", "sigma_ratio"]), errors="coerce"
)
resource["GTEx_low_confidence_flag"] = get_col(hcm, ["GTEx_low_confidence_flag"], np.nan)
resource["GTEx_adjustment_status"] = get_col(hcm, ["GTEx_adjustment_status"], np.nan)

resource["n_cohorts_available"] = pd.to_numeric(get_col(hcm, ["n_cohorts_available", "n_cohorts", "cohorts"]), errors="coerce")
resource["D_g_direction_agreement"] = pd.to_numeric(get_col(hcm, ["D_g", "D_g_direction_agreement", "direction_agreement"]), errors="coerce")
resource["S_g_effect_size_consistency"] = pd.to_numeric(get_col(hcm, ["S_g", "S_g_effect_size_consistency", "effect_size_consistency"]), errors="coerce")
resource["R_g_statistical_reproducibility"] = pd.to_numeric(get_col(hcm, ["R_g", "R_g_statistical_reproducibility", "statistical_reproducibility"]), errors="coerce")

# Add resource interpretation field
def interpret_row(row):
    tier = str(row.get("MCI_tier", "")).upper()
    gtex_flag = row.get("GTEx_low_confidence_flag", False)
    ratio = row.get("sigma_disease_to_GTEx_ratio", np.nan)

    if pd.isna(row.get("MCI")):
        return "Insufficient evidence for primary MCI interpretation."
    if tier == "HIGH" and (gtex_flag is True or str(gtex_flag).lower() == "true"):
        return "High cross-cohort transcript concordance, but disease variability does not exceed GTEx LV baseline. Interpret as transcript-concordant but GTEx-low-confidence."
    if tier == "HIGH":
        return "High cross-cohort transcript concordance. Candidate for prioritized orthogonal validation."
    if tier == "MODERATE":
        return "Moderate transcript concordance. Candidate for conditional follow-up with additional cohort or orthogonal evidence."
    if tier == "UNSTABLE":
        return "Unstable transcript-level evidence across current HCM cohorts. Do not treat bulk transcript signal as reproducible without additional validation."
    if "INSUFFICIENT" in tier:
        return "Insufficient cohort coverage for stable concordance interpretation."
    return "Resource entry available. Interpret with MCI tier, bootstrap interval, and GTEx confidence status."

resource["resource_interpretation"] = resource.apply(interpret_row, axis=1)

# -------------------------
# 4. Attach DCM availability flags if DCM files exist
# -------------------------

resource["has_DCM_generalization_entry"] = False
resource["DCM_MCI_if_available"] = np.nan
resource["DCM_tier_if_available"] = np.nan

dcm_loaded_any = False

for path in dcm_paths[:10]:
    dcm = read_csv_safe(path)
    if dcm is None or dcm.empty:
        continue

    # find gene column
    dcm_gene_cols = [c for c in dcm.columns if "gene" in c.lower() and ("symbol" in c.lower() or c.lower() == "gene")]
    if not dcm_gene_cols:
        continue
    dcm_gene_col = dcm_gene_cols[0]
    dcm["gene_symbol_std"] = standardize_gene_symbol(dcm[dcm_gene_col])

    # find MCI column
    dcm_mci_cols = [c for c in dcm.columns if c.upper() == "MCI" or "DCM_MCI" in c.upper()]
    dcm_tier_cols = [c for c in dcm.columns if "tier" in c.lower()]

    if dcm_mci_cols:
        dcm_small = dcm[["gene_symbol_std", dcm_mci_cols[0]]].copy()
        dcm_small = dcm_small.dropna(subset=["gene_symbol_std"]).drop_duplicates("gene_symbol_std")
        dcm_small.columns = ["gene_symbol", "DCM_MCI_from_file"]

        resource = resource.merge(dcm_small, on="gene_symbol", how="left")
        resource["DCM_MCI_if_available"] = resource["DCM_MCI_if_available"].combine_first(resource["DCM_MCI_from_file"])
        resource.drop(columns=["DCM_MCI_from_file"], inplace=True)
        resource["has_DCM_generalization_entry"] = resource["DCM_MCI_if_available"].notna()
        dcm_loaded_any = True

    if dcm_tier_cols:
        dcm_tier = dcm[["gene_symbol_std", dcm_tier_cols[0]]].copy()
        dcm_tier = dcm_tier.dropna(subset=["gene_symbol_std"]).drop_duplicates("gene_symbol_std")
        dcm_tier.columns = ["gene_symbol", "DCM_tier_from_file"]
        resource = resource.merge(dcm_tier, on="gene_symbol", how="left")
        resource["DCM_tier_if_available"] = resource["DCM_tier_if_available"].combine_first(resource["DCM_tier_from_file"])
        resource.drop(columns=["DCM_tier_from_file"], inplace=True)

print("\nDCM values attached:", dcm_loaded_any)

# -------------------------
# 5. Add resource version fields
# -------------------------

version_date = datetime.datetime.utcnow().strftime("%Y-%m-%d")
resource["resource_version"] = "v0.1-resource-reframe"
resource["resource_release_date_utc"] = version_date
resource["primary_data_type"] = "bulk_RNA_expression"
resource["resource_status"] = "browser_ready_table_candidate"

# Sort by disease group and MCI descending
resource = resource.sort_values(["disease_group", "MCI"], ascending=[True, False])

# -------------------------
# 6. Save master table and data dictionary
# -------------------------

master_path = resource_table_dir / "MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_1.csv"
resource.to_csv(master_path, index=False)

data_dictionary = pd.DataFrame([
    {"column": "gene_symbol", "description": "HGNC-style gene symbol used as primary resource key."},
    {"column": "disease_group", "description": "Disease group for the resource row. Current master table is HCM-primary."},
    {"column": "source_resource_layer", "description": "Indicates whether row came from primary HCM, DCM generalization, validation, or future scRNA module."},
    {"column": "stratum", "description": "Pre-specified biological mechanism group where available."},
    {"column": "MCI", "description": "Primary Molecular Concordance Index using direction agreement, effect-size consistency, and statistical reproducibility."},
    {"column": "Adj_MCI", "description": "GTEx-adjusted MCI using disease-to-GTEx variability ratio."},
    {"column": "MCI_tier", "description": "Primary tier assignment: HIGH, MODERATE, UNSTABLE, or insufficient coverage."},
    {"column": "bootstrap_majority_tier", "description": "Tier most frequently assigned across bootstrap resampling."},
    {"column": "MCI_CI95_lower", "description": "Lower percentile bootstrap 95 percent interval for MCI."},
    {"column": "MCI_CI95_upper", "description": "Upper percentile bootstrap 95 percent interval for MCI."},
    {"column": "sigma_disease", "description": "Standard deviation of HCM log2FC values across primary disease cohorts."},
    {"column": "sigma_GTEx", "description": "Standard deviation of GTEx v8 Heart - Left Ventricle log2(TPM + 1) expression."},
    {"column": "sigma_disease_to_GTEx_ratio", "description": "Disease variability divided by GTEx baseline variability."},
    {"column": "GTEx_low_confidence_flag", "description": "True if disease variability does not exceed GTEx baseline variability."},
    {"column": "n_cohorts_available", "description": "Number of disease cohorts contributing to MCI score."},
    {"column": "resource_interpretation", "description": "Plain-language interpretation for browser/database use."},
    {"column": "has_DCM_generalization_entry", "description": "Whether a DCM generalization value could be attached from available files."},
    {"column": "resource_version", "description": "Version label for this resource table."},
])

dict_path = resource_table_dir / "MCI_CARDIOMYOPATHY_RESOURCE_DATA_DICTIONARY_v0_1.csv"
data_dictionary.to_csv(dict_path, index=False)

# -------------------------
# 7. Save QC summary
# -------------------------

summary = {
    "step": "STEP_3_RESOURCE_MASTER_TABLE",
    "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "hcm_source_file": str(hcm_final_path.relative_to(REPO_DIR)),
    "hcm_source_shape": list(hcm.shape),
    "resource_master_table": str(master_path.relative_to(REPO_DIR)),
    "resource_master_shape": list(resource.shape),
    "n_genes": int(resource["gene_symbol"].nunique()),
    "tier_counts": resource["MCI_tier"].astype(str).value_counts(dropna=False).to_dict(),
    "gtex_low_confidence_counts": resource["GTEx_low_confidence_flag"].astype(str).value_counts(dropna=False).to_dict(),
    "dcm_files_detected": [str(p.relative_to(REPO_DIR)) for p in dcm_paths[:20]],
    "validation_files_detected": [str(p.relative_to(REPO_DIR)) for p in validation_paths[:20]],
    "dcm_values_attached": bool(dcm_loaded_any),
}

summary_path = qc_dir / "STEP3_RESOURCE_MASTER_TABLE_SUMMARY.json"
summary_path.write_text(json.dumps(summary, indent=2))

# -------------------------
# 8. Display useful preview
# -------------------------

print("\nSTEP 3 COMPLETE.")
print("Saved master table:", master_path)
print("Saved data dictionary:", dict_path)
print("Saved QC summary:", summary_path)

print("\nResource table shape:", resource.shape)
print("\nTier counts:")
print(resource["MCI_tier"].value_counts(dropna=False))

print("\nGTEx low-confidence flag counts:")
print(resource["GTEx_low_confidence_flag"].value_counts(dropna=False))

print("\nTop 12 resource rows:")
display_cols = [
    "gene_symbol", "disease_group", "stratum", "MCI", "Adj_MCI", "MCI_tier",
    "MCI_CI95_lower", "MCI_CI95_upper", "sigma_disease_to_GTEx_ratio",
    "GTEx_low_confidence_flag", "resource_interpretation"
]
display(resource[display_cols].head(12))


Final HCM MCI + GTEx + bootstrap table: found 6 candidate files
  1. results/quality_control/mci_scores/TASK4_CODE25_HCM_MCI_bootstrap_summary.csv
  2. results/quality_control/mci_scores/TASK4_CODE25_HCM_MCI_bootstrap_draws_long.csv
  3. results/quality_control/mci_scores/TASK4_CODE25_HCM_MCI_bootstrap_majority_tier_counts.csv
  4. results/mci_scores/TASK6_FINAL_HCM_MCI_WITH_GTEx_AND_BOOTSTRAP_CI.csv
  5. results/mci_scores/TASK4_FINAL_REAL_HCM_MCI_SCORE_TABLE_WITH_BOOTSTRAP_CI.csv
  6. results/mci_scores/TASK4_CODE25_REAL_HCM_MCI_scores_primary_with_bootstrap_CI.csv
Selected for Final HCM MCI + GTEx + bootstrap table: results/quality_control/mci_scores/TASK4_CODE25_HCM_MCI_bootstrap_summary.csv

DCM candidate tables found: 16
  1. results/quality_control/dcm_generalization/TASK8_DCM_gene_coverage_before_MCI.csv
  2. results/quality_control/dcm_generalization/TASK8_DCM_GENERALIZATION_FILE_MANIFEST.csv
  3. results/quality_control/dcm_generalization/TASK8_BOOTSTRAP_DCM_MCI_CI_all_draws

/tmp/ipykernel_2654/285610975.py:229: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  version_date = datetime.datetime.utcnow().strftime("%Y-%m-%d")
/tmp/ipykernel_2654/285610975.py:275: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",


,gene_symbol,disease_group,stratum,MCI,Adj_MCI,MCI_tier,MCI_CI95_lower,MCI_CI95_upper,sigma_disease_to_GTEx_ratio,GTEx_low_confidence_flag,resource_interpretation
24,MYH6,HCM,NaN,0.968060,NaN,HIGH,0.961784,1.0,NaN,NaN,High cross-cohort transcript concordance. Cand...
44,TNNI3,HCM,NaN,0.961721,NaN,HIGH,0.953660,1.0,NaN,NaN,High cross-cohort transcript concordance. Cand...
12,GLA,HCM,NaN,0.960029,NaN,HIGH,0.950944,1.0,NaN,NaN,High cross-cohort transcript concordance. Cand...
1,ACTN2,HCM,NaN,0.934792,NaN,HIGH,0.917719,1.0,NaN,NaN,High cross-cohort transcript concordance. Cand...
42,TMEM43,HCM,NaN,0.929496,NaN,HIGH,0.904270,1.0,NaN,NaN,High cross-cohort transcript concordance. Cand...
41,TINF2,HCM,NaN,0.904054,NaN,HIGH,0.858822,1.0,NaN,NaN,High cross-cohort transcript concordance. Cand...
14,KCNH2,HCM,NaN,0.881352,NaN,HIGH,0.803704,1.0,NaN,NaN,High cross-cohort transcript concordance. Cand...
3,BAG3,HCM,NaN,0.828181,NaN,HIGH,0.704683,1.0,NaN,NaN,High cross-cohort transcript concordance. Cand...
47,TRIM63,HCM,NaN,0.819793,NaN,HIGH,0.744755,1.0,NaN,NaN,High cross-cohort transcript concordance. Cand...
36,RBM20,HCM,NaN,0.803574,NaN,HIGH,0.652171,1.0,NaN,NaN,High cross-cohort transcript concordance. Cand...


In [ ]:
# =========================
# STEP 3B: Fix master table source file and rebuild with GTEx fields
# =========================

from pathlib import Path
import pandas as pd
import numpy as np
import json, datetime

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")
assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

resource_table_dir = REPO_DIR / "results" / "resource_tables"
qc_dir = REPO_DIR / "results" / "quality_control" / "resource_tables"
resource_table_dir.mkdir(parents=True, exist_ok=True)
qc_dir.mkdir(parents=True, exist_ok=True)

# Force correct final HCM table
hcm_final_path = REPO_DIR / "results" / "mci_scores" / "TASK6_FINAL_HCM_MCI_WITH_GTEx_AND_BOOTSTRAP_CI.csv"

if not hcm_final_path.exists():
    raise FileNotFoundError(f"Correct final HCM table not found: {hcm_final_path}")

hcm = pd.read_csv(hcm_final_path)

print("Loaded correct HCM table:")
print(hcm_final_path.relative_to(REPO_DIR))
print("Shape:", hcm.shape)
print("Columns:")
print(list(hcm.columns))

def standardize_gene_symbol(series):
    return (
        series.astype(str)
        .str.strip()
        .str.upper()
        .replace({"NAN": np.nan, "NONE": np.nan, "": np.nan})
    )

def pick_col(df, candidates, required=False):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"None of these required columns found: {candidates}")
    return None

def get_series(df, candidates, default=np.nan):
    col = pick_col(df, candidates, required=False)
    if col is None:
        return default
    return df[col]

gene_col = pick_col(hcm, ["gene_symbol", "Gene", "gene"], required=True)

# Build corrected resource table
resource = pd.DataFrame()
resource["gene_symbol"] = standardize_gene_symbol(hcm[gene_col])
resource["disease_group"] = "HCM"
resource["source_resource_layer"] = "primary_HCM_bulk_RNA_resource"

resource["stratum"] = get_series(hcm, ["stratum", "mechanism_group", "gene_stratum"])
resource["is_prespecified_h4_gene"] = get_series(hcm, ["is_prespecified_h4_gene"], False)

resource["MCI"] = pd.to_numeric(
    get_series(hcm, ["MCI", "MCI_original_from_step5", "point_MCI", "MCI_recomputed_point"]),
    errors="coerce"
)

resource["Adj_MCI"] = pd.to_numeric(
    get_series(hcm, ["Adj_MCI", "GTEx_adjusted_MCI", "adjusted_MCI"]),
    errors="coerce"
)

resource["MCI_tier"] = get_series(hcm, ["MCI_tier", "tier", "primary_MCI_tier"])
resource["bootstrap_majority_tier"] = get_series(
    hcm,
    ["bootstrap_majority_tier", "bootstrap_tier_majority", "majority_tier"]
)

resource["MCI_CI95_lower"] = pd.to_numeric(
    get_series(hcm, ["MCI_CI95_lower", "CI_lower", "bootstrap_CI_lower"]),
    errors="coerce"
)

resource["MCI_CI95_upper"] = pd.to_numeric(
    get_series(hcm, ["MCI_CI95_upper", "CI_upper", "bootstrap_CI_upper"]),
    errors="coerce"
)

resource["MCI_CI95_width"] = resource["MCI_CI95_upper"] - resource["MCI_CI95_lower"]

resource["bootstrap_median_MCI"] = pd.to_numeric(
    get_series(hcm, ["bootstrap_median_MCI, MCI_bootstrap_median", "MCI_bootstrap_median"]),
    errors="coerce"
)

resource["bootstrap_prob_HIGH"] = pd.to_numeric(
    get_series(hcm, ["bootstrap_prob_HIGH"]),
    errors="coerce"
)

resource["bootstrap_prob_MODERATE"] = pd.to_numeric(
    get_series(hcm, ["bootstrap_prob_MODERATE"]),
    errors="coerce"
)

resource["bootstrap_prob_UNSTABLE"] = pd.to_numeric(
    get_series(hcm, ["bootstrap_prob_UNSTABLE"]),
    errors="coerce"
)

resource["sigma_disease"] = pd.to_numeric(
    get_series(hcm, ["sigma_disease"]),
    errors="coerce"
)

resource["sigma_GTEx"] = pd.to_numeric(
    get_series(hcm, ["sigma_GTEx"]),
    errors="coerce"
)

resource["sigma_disease_to_GTEx_ratio"] = pd.to_numeric(
    get_series(hcm, ["sigma_disease_to_GTEx_ratio", "sigma_ratio"]),
    errors="coerce"
)

resource["GTEx_low_confidence_flag"] = get_series(hcm, ["GTEx_low_confidence_flag"])
resource["GTEx_adjustment_status"] = get_series(hcm, ["GTEx_adjustment_status"])

resource["n_cohorts_available"] = pd.to_numeric(
    get_series(hcm, ["n_cohorts_available", "n_hcm_cohorts_available", "n_cohorts"]),
    errors="coerce"
)

resource["D_g_direction_agreement"] = pd.to_numeric(
    get_series(hcm, ["D_g", "D_g_direction_agreement", "direction_agreement"]),
    errors="coerce"
)

resource["S_g_effect_size_consistency"] = pd.to_numeric(
    get_series(hcm, ["S_g", "S_g_effect_size_consistency", "effect_size_consistency"]),
    errors="coerce"
)

resource["R_g_statistical_reproducibility"] = pd.to_numeric(
    get_series(hcm, ["R_g", "R_g_statistical_reproducibility", "statistical_reproducibility"]),
    errors="coerce"
)

# Clean boolean flags
resource["GTEx_low_confidence_flag"] = (
    resource["GTEx_low_confidence_flag"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False, "1": True, "0": False, "nan": np.nan})
)

def interpret_row(row):
    tier = str(row.get("MCI_tier", "")).upper()
    gtex_flag = row.get("GTEx_low_confidence_flag", np.nan)
    ratio = row.get("sigma_disease_to_GTEx_ratio", np.nan)

    if pd.isna(row.get("MCI")):
        return "Insufficient evidence for primary MCI interpretation."

    if tier == "HIGH" and gtex_flag is True:
        return "High cross-cohort transcript concordance, but disease variability does not exceed GTEx LV baseline. Interpret as transcript-concordant but GTEx-low-confidence."

    if tier == "HIGH":
        return "High cross-cohort transcript concordance. Candidate for prioritized orthogonal validation."

    if tier == "MODERATE" and gtex_flag is True:
        return "Moderate transcript concordance with GTEx-low-confidence status. Requires additional cohort, cell-type, or orthogonal evidence."

    if tier == "MODERATE":
        return "Moderate transcript concordance. Candidate for conditional follow-up with additional cohort or orthogonal evidence."

    if tier == "UNSTABLE":
        return "Unstable transcript-level evidence across current HCM cohorts. Do not treat bulk transcript signal as reproducible without additional validation."

    if "INSUFFICIENT" in tier:
        return "Insufficient cohort coverage for stable concordance interpretation."

    return "Resource entry available. Interpret with MCI tier, bootstrap interval, and GTEx confidence status."

resource["resource_interpretation"] = resource.apply(interpret_row, axis=1)

# Attach DCM values from the correct final DCM table if available
resource["has_DCM_generalization_entry"] = False
resource["DCM_MCI_if_available"] = np.nan
resource["DCM_tier_if_available"] = np.nan

dcm_final_path = REPO_DIR / "results" / "mci_scores" / "TASK8_FINAL_DCM_MCI_GENERALIZATION_WITH_BOOTSTRAP_CI.csv"

if dcm_final_path.exists():
    dcm = pd.read_csv(dcm_final_path)
    print("\nLoaded DCM table:")
    print(dcm_final_path.relative_to(REPO_DIR))
    print("DCM shape:", dcm.shape)

    dcm_gene_col = pick_col(dcm, ["gene_symbol", "Gene", "gene"], required=True)
    dcm["gene_symbol"] = standardize_gene_symbol(dcm[dcm_gene_col])

    dcm_mci_col = pick_col(dcm, ["MCI", "DCM_MCI", "point_MCI"], required=False)
    dcm_tier_col = pick_col(dcm, ["MCI_tier", "tier", "DCM_MCI_tier"], required=False)

    dcm_small = dcm[["gene_symbol"]].copy()

    if dcm_mci_col:
        dcm_small["DCM_MCI_if_available"] = pd.to_numeric(dcm[dcm_mci_col], errors="coerce")

    if dcm_tier_col:
        dcm_small["DCM_tier_if_available"] = dcm[dcm_tier_col]

    dcm_small = dcm_small.drop_duplicates("gene_symbol")

    resource = resource.drop(columns=["DCM_MCI_if_available", "DCM_tier_if_available"])
    resource = resource.merge(dcm_small, on="gene_symbol", how="left")
    resource["has_DCM_generalization_entry"] = resource["DCM_MCI_if_available"].notna()

else:
    print("\nWARNING: Final DCM table not found. DCM fields left blank.")

# Add resource version metadata
resource["resource_version"] = "v0.2-resource-reframe-corrected-source"
resource["resource_release_date_utc"] = datetime.datetime.now(datetime.UTC).strftime("%Y-%m-%d")
resource["primary_data_type"] = "bulk_RNA_expression"
resource["resource_status"] = "browser_ready_table_candidate"

# Sort
resource = resource.sort_values(["disease_group", "MCI"], ascending=[True, False])

# Save corrected master table
master_path = resource_table_dir / "MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv"
resource.to_csv(master_path, index=False)

# Save data dictionary
data_dictionary = pd.DataFrame([
    {"column": "gene_symbol", "description": "HGNC-style gene symbol used as the primary resource key."},
    {"column": "disease_group", "description": "Primary disease context for the row."},
    {"column": "source_resource_layer", "description": "Resource layer from which the row was generated."},
    {"column": "stratum", "description": "Pre-specified biological mechanism group where available."},
    {"column": "MCI", "description": "Primary Molecular Concordance Index."},
    {"column": "Adj_MCI", "description": "GTEx-adjusted MCI."},
    {"column": "MCI_tier", "description": "Primary MCI tier classification."},
    {"column": "bootstrap_majority_tier", "description": "Most frequent tier across bootstrap resampling."},
    {"column": "MCI_CI95_lower", "description": "Lower bootstrap 95 percent CI bound."},
    {"column": "MCI_CI95_upper", "description": "Upper bootstrap 95 percent CI bound."},
    {"column": "sigma_disease", "description": "Disease-side cross-cohort log2FC variability."},
    {"column": "sigma_GTEx", "description": "GTEx v8 Heart - Left Ventricle normal expression variability."},
    {"column": "sigma_disease_to_GTEx_ratio", "description": "Disease variability divided by GTEx normal baseline variability."},
    {"column": "GTEx_low_confidence_flag", "description": "True where disease variability does not exceed GTEx baseline variability."},
    {"column": "GTEx_adjustment_status", "description": "Interpretive GTEx adjustment label."},
    {"column": "resource_interpretation", "description": "Plain-language interpretation for database/browser display."},
    {"column": "DCM_MCI_if_available", "description": "DCM generalization MCI value, where available."},
    {"column": "DCM_tier_if_available", "description": "DCM tier, where available."},
])
dict_path = resource_table_dir / "MCI_CARDIOMYOPATHY_RESOURCE_DATA_DICTIONARY_v0_2.csv"
data_dictionary.to_csv(dict_path, index=False)

# QC summary
summary = {
    "step": "STEP_3B_CORRECTED_RESOURCE_MASTER_TABLE",
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "correct_hcm_source_file": str(hcm_final_path.relative_to(REPO_DIR)),
    "hcm_source_shape": list(hcm.shape),
    "dcm_source_file": str(dcm_final_path.relative_to(REPO_DIR)) if dcm_final_path.exists() else None,
    "resource_master_table": str(master_path.relative_to(REPO_DIR)),
    "resource_master_shape": list(resource.shape),
    "n_genes": int(resource["gene_symbol"].nunique()),
    "tier_counts": resource["MCI_tier"].astype(str).value_counts(dropna=False).to_dict(),
    "gtex_low_confidence_counts": resource["GTEx_low_confidence_flag"].astype(str).value_counts(dropna=False).to_dict(),
    "n_with_sigma_ratio": int(resource["sigma_disease_to_GTEx_ratio"].notna().sum()),
    "n_with_dcm_entry": int(resource["has_DCM_generalization_entry"].sum()),
}

summary_path = qc_dir / "STEP3B_CORRECTED_RESOURCE_MASTER_TABLE_SUMMARY.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("\nSTEP 3B COMPLETE.")
print("Corrected master table:", master_path)
print("Corrected data dictionary:", dict_path)
print("QC summary:", summary_path)

print("\nTier counts:")
print(resource["MCI_tier"].value_counts(dropna=False))

print("\nGTEx low-confidence counts:")
print(resource["GTEx_low_confidence_flag"].value_counts(dropna=False))

print("\nRows with sigma ratio:", resource["sigma_disease_to_GTEx_ratio"].notna().sum())
print("Rows with DCM entry:", resource["has_DCM_generalization_entry"].sum())

print("\nTop 12 corrected resource rows:")
display_cols = [
    "gene_symbol", "stratum", "MCI", "Adj_MCI", "MCI_tier",
    "MCI_CI95_lower", "MCI_CI95_upper",
    "sigma_disease_to_GTEx_ratio", "GTEx_low_confidence_flag",
    "DCM_MCI_if_available", "resource_interpretation"
]
display(resource[display_cols].head(12))

Loaded correct HCM table:
results/mci_scores/TASK6_FINAL_HCM_MCI_WITH_GTEx_AND_BOOTSTRAP_CI.csv
Shape: (49, 72)
Columns:
['gene_symbol', 'disease_group', 'stratum', 'sources', 'is_prespecified_h4_gene', 'present_in_strict_clinvar_filter', 'max_n_plp_variants', 'max_review_weight', 'n_hcm_cohorts_available', 'cohorts_available', 'mean_log2FC', 'sd_log2FC', 'min_FDR', 'n_FDR_lt_0_05', 'D_g_direction_agreement', 'S_g_effect_size_consistency', 'R_g_statistical_reproducibility', 'MCI', 'MCI_tier', 'bootstrap_tier_majority', 'tier_stability_note', 'CI_crosses_HIGH_threshold_0_70', 'CI_crosses_MODERATE_threshold_0_45', 'mean_log2FC_rounded', 'sd_log2FC_rounded', 'min_FDR_rounded', 'D_g_direction_agreement_rounded', 'S_g_effect_size_consistency_rounded', 'R_g_statistical_reproducibility_rounded', 'MCI_rounded', 'MCI_CI95_lower_rounded', 'MCI_CI95_upper_rounded', 'MCI_CI95_width_rounded', 'bootstrap_prob_HIGH_rounded', 'bootstrap_prob_MODERATE_or_HIGH_rounded', 'bootstrap_prob_UNSTABLE_rounded'

,gene_symbol,stratum,MCI,Adj_MCI,MCI_tier,MCI_CI95_lower,MCI_CI95_upper,sigma_disease_to_GTEx_ratio,GTEx_low_confidence_flag,DCM_MCI_if_available,resource_interpretation
0,MYH6,NaN,0.968060,1.184359,HIGH,0.961784,1.0,0.167511,True,NaN,"High cross-cohort transcript concordance, but ..."
1,TNNI3,HCM_sarcomeric,0.961721,1.050513,HIGH,0.953660,1.0,0.066088,True,0.846115,"High cross-cohort transcript concordance, but ..."
2,GLA,NaN,0.960029,1.185175,HIGH,0.950944,1.0,0.176515,True,NaN,"High cross-cohort transcript concordance, but ..."
3,ACTN2,NaN,0.934792,1.000238,HIGH,0.917719,1.0,0.049725,True,0.728090,"High cross-cohort transcript concordance, but ..."
4,TMEM43,NaN,0.929496,1.213745,HIGH,0.904270,1.0,0.236112,True,0.998804,"High cross-cohort transcript concordance, but ..."
5,TINF2,NaN,0.904054,1.199507,HIGH,0.858822,1.0,0.254236,True,NaN,"High cross-cohort transcript concordance, but ..."
6,KCNH2,NaN,0.881352,1.128039,HIGH,0.803704,1.0,0.214108,True,NaN,"High cross-cohort transcript concordance, but ..."
7,BAG3,HCM_non_sarcomeric,0.828181,0.976573,HIGH,0.704683,1.0,0.132238,True,0.692754,"High cross-cohort transcript concordance, but ..."
8,TRIM63,NaN,0.819793,0.832012,HIGH,0.744755,1.0,0.010384,True,NaN,"High cross-cohort transcript concordance, but ..."
9,RBM20,HCM_non_sarcomeric,0.803574,1.079434,HIGH,0.652171,1.0,0.268648,True,NaN,"High cross-cohort transcript concordance, but ..."


In [ ]:
# =========================
# STEP 4: Create GitHub-facing resource README and resource documentation
# =========================

from pathlib import Path
import pandas as pd
import json, datetime, textwrap

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")
assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

resource_table_path = REPO_DIR / "results" / "resource_tables" / "MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv"
assert resource_table_path.exists(), f"Resource table not found: {resource_table_path}"

resource = pd.read_csv(resource_table_path)

docs_dir = REPO_DIR / "docs" / "resource_framing"
manuscript_dir = REPO_DIR / "manuscript" / "resource_paper"
metadata_dir = REPO_DIR / "metadata"

docs_dir.mkdir(parents=True, exist_ok=True)
manuscript_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

n_genes = resource["gene_symbol"].nunique()
tier_counts = resource["MCI_tier"].value_counts(dropna=False).to_dict()
gtex_counts = resource["GTEx_low_confidence_flag"].value_counts(dropna=False).to_dict()
n_sigma = int(resource["sigma_disease_to_GTEx_ratio"].notna().sum())
n_dcm = int(resource["has_DCM_generalization_entry"].sum())

high_n = int(tier_counts.get("HIGH", 0))
moderate_n = int(tier_counts.get("MODERATE", 0))
unstable_n = int(tier_counts.get("UNSTABLE", 0))
insufficient_n = int(tier_counts.get("INSUFFICIENT_COHORT_COVERAGE", 0))

gtex_low_n = int(gtex_counts.get(True, 0))
gtex_not_low_n = int(gtex_counts.get(False, 0))

# -------------------------
# 1. Create resource landing page README
# -------------------------

resource_readme = f"""# Molecular Concordance Index Cardiomyopathy Resource

## Overview

This repository contains the Molecular Concordance Index (MCI) cardiomyopathy resource, a reproducible transcriptomic evidence-auditing framework for ClinVar-annotated hypertrophic cardiomyopathy (HCM) and dilated cardiomyopathy (DCM) genes.

The goal of this resource is not to claim that every clinically annotated cardiomyopathy gene has reproducible bulk-transcriptomic evidence. Instead, the resource makes that question explicit and queryable.

MCI separates four evidence layers that are often conflated:

1. clinical pathogenicity annotation,
2. cross-cohort transcriptomic reproducibility,
3. normal left-ventricle expression variability,
4. independent validation or convergence evidence.

The central resource table allows users to inspect whether a gene is transcript-concordant, transcript-unstable, GTEx-low-confidence, DCM-generalizable, or insufficiently covered by current public data.

## Current resource version

Version: `v0.2-resource-reframe-corrected-source`

Primary master table:

`results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv`

Data dictionary:

`results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_DATA_DICTIONARY_v0_2.csv`

## Current resource content

The current HCM-primary resource contains:

- {n_genes} ClinVar-annotated HCM genes
- {high_n} HIGH MCI genes
- {moderate_n} MODERATE MCI genes
- {unstable_n} UNSTABLE MCI genes
- {insufficient_n} genes with insufficient cohort coverage
- {n_sigma} genes with disease-to-GTEx variability ratios
- {gtex_low_n} genes flagged as GTEx-low-confidence
- {gtex_not_low_n} genes whose disease variability exceeded GTEx baseline or were not low-confidence
- {n_dcm} genes with attached DCM generalization entries

## How to interpret MCI

MCI is a composite transcriptomic concordance score:

`MCI_g = 0.40 * D_g + 0.35 * S_g + 0.25 * R_g`

where:

- `D_g` = direction agreement across cohorts
- `S_g` = effect-size consistency across cohorts
- `R_g` = statistical reproducibility across cohorts

Tier labels:

- `HIGH`: MCI >= 0.70
- `MODERATE`: 0.45 <= MCI < 0.70
- `UNSTABLE`: MCI < 0.45
- `INSUFFICIENT_COHORT_COVERAGE`: fewer than two eligible cohorts

## GTEx baseline interpretation

The GTEx layer compares disease-associated variability against normal human left-ventricle expression variability.

`ratio = sigma_disease / sigma_GTEx`

If the ratio is less than or equal to 1.0, the gene is flagged as GTEx-low-confidence. This does not mean the gene is biologically irrelevant. It means the disease-associated bulk-transcriptomic variability does not exceed normal GTEx left-ventricle baseline variability in the current data.

This distinction is important because a gene may be clinically pathogenic and transcript-concordant while still requiring orthogonal support from genotype-stratified cohorts, single-cell analysis, proteomics, or functional validation.

## Main claim boundary

This resource does not claim that MCI proves causality, clinical actionability, or therapeutic validity.

The resource is designed for evidence auditing and target-prioritization support. It should be used to identify genes whose transcript-level evidence is reproducible, unstable, baseline-confounded, disease-context-specific, or underpowered.

## Manuscript framing

This project is best framed as a database/resource paper rather than as a high-impact biological discovery paper.

Recommended manuscript title:

**Molecular Concordance Index: a reproducible transcriptomic resource for evaluating ClinVar-annotated cardiomyopathy genes across human HCM and DCM cohorts**

## Repository structure

```text
results/resource_tables/
  MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv
  MCI_CARDIOMYOPATHY_RESOURCE_DATA_DICTIONARY_v0_2.csv

manuscript/resource_paper/
  00_RESOURCE_PAPER_POSITIONING.md
  01_REVISED_RESOURCE_ABSTRACT.md
  02_REVISED_CONTRIBUTIONS.md
  03_RESOURCE_MANUSCRIPT_OUTLINE.md

docs/resource_framing/
  CLAIM_BOUNDARY_STATEMENT.md
  RESOURCE_TABLE_USAGE_GUIDE.md

scripts/scrna_contextualization/
  Placeholder for future public single-cell RNA-seq contextualization module.

shiny_app/
  Browser files or future browser outputs.

SyntaxError: incomplete input (3596080716.py, line 43)

In [ ]:
# =========================
# STEP 4: Create GitHub-facing resource README and resource documentation
# =========================

from pathlib import Path
import pandas as pd
import json, datetime, textwrap

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")
assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

resource_table_path = REPO_DIR / "results" / "resource_tables" / "MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv"
assert resource_table_path.exists(), f"Resource table not found: {resource_table_path}"

resource = pd.read_csv(resource_table_path)

docs_dir = REPO_DIR / "docs" / "resource_framing"
manuscript_dir = REPO_DIR / "manuscript" / "resource_paper"
metadata_dir = REPO_DIR / "metadata"

docs_dir.mkdir(parents=True, exist_ok=True)
manuscript_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

n_genes = resource["gene_symbol"].nunique()
tier_counts = resource["MCI_tier"].value_counts(dropna=False).to_dict()
gtex_counts = resource["GTEx_low_confidence_flag"].value_counts(dropna=False).to_dict()
n_sigma = int(resource["sigma_disease_to_GTEx_ratio"].notna().sum())
n_dcm = int(resource["has_DCM_generalization_entry"].sum())

high_n = int(tier_counts.get("HIGH", 0))
moderate_n = int(tier_counts.get("MODERATE", 0))
unstable_n = int(tier_counts.get("UNSTABLE", 0))
insufficient_n = int(tier_counts.get("INSUFFICIENT_COHORT_COVERAGE", 0))

gtex_low_n = int(gtex_counts.get(True, 0))
gtex_not_low_n = int(gtex_counts.get(False, 0))

# -------------------------
# 1. Create resource landing page README
# -------------------------

resource_readme = f"""# Molecular Concordance Index Cardiomyopathy Resource

## Overview

This repository contains the Molecular Concordance Index (MCI) cardiomyopathy resource, a reproducible transcriptomic evidence-auditing framework for ClinVar-annotated hypertrophic cardiomyopathy (HCM) and dilated cardiomyopathy (DCM) genes.

The goal of this resource is not to claim that every clinically annotated cardiomyopathy gene has reproducible bulk-transcriptomic evidence. Instead, the resource makes that question explicit and queryable.

MCI separates four evidence layers that are often conflated:

1. clinical pathogenicity annotation,
2. cross-cohort transcriptomic reproducibility,
3. normal left-ventricle expression variability,
4. independent validation or convergence evidence.

The central resource table allows users to inspect whether a gene is transcript-concordant, transcript-unstable, GTEx-low-confidence, DCM-generalizable, or insufficiently covered by current public data.

## Current resource version

Version: `v0.2-resource-reframe-corrected-source`

Primary master table:

`results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv`

Data dictionary:

`results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_DATA_DICTIONARY_v0_2.csv`

## Current resource content

The current HCM-primary resource contains:

- {n_genes} ClinVar-annotated HCM genes
- {high_n} HIGH MCI genes
- {moderate_n} MODERATE MCI genes
- {unstable_n} UNSTABLE MCI genes
- {insufficient_n} genes with insufficient cohort coverage
- {n_sigma} genes with disease-to-GTEx variability ratios
- {gtex_low_n} genes flagged as GTEx-low-confidence
- {gtex_not_low_n} genes whose disease variability exceeded GTEx baseline or were not low-confidence
- {n_dcm} genes with attached DCM generalization entries

## How to interpret MCI

MCI is a composite transcriptomic concordance score:

`MCI_g = 0.40 * D_g + 0.35 * S_g + 0.25 * R_g`

where:

- `D_g` = direction agreement across cohorts
- `S_g` = effect-size consistency across cohorts
- `R_g` = statistical reproducibility across cohorts

Tier labels:

- `HIGH`: MCI >= 0.70
- `MODERATE`: 0.45 <= MCI < 0.70
- `UNSTABLE`: MCI < 0.45
- `INSUFFICIENT_COHORT_COVERAGE`: fewer than two eligible cohorts

## GTEx baseline interpretation

The GTEx layer compares disease-associated variability against normal human left-ventricle expression variability.

`ratio = sigma_disease / sigma_GTEx`

If the ratio is less than or equal to 1.0, the gene is flagged as GTEx-low-confidence. This does not mean the gene is biologically irrelevant. It means the disease-associated bulk-transcriptomic variability does not exceed normal GTEx left-ventricle baseline variability in the current data.

This distinction is important because a gene may be clinically pathogenic and transcript-concordant while still requiring orthogonal support from genotype-stratified cohorts, single-cell analysis, proteomics, or functional validation.

## Main claim boundary

This resource does not claim that MCI proves causality, clinical actionability, or therapeutic validity.

The resource is designed for evidence auditing and target-prioritization support. It should be used to identify genes whose transcript-level evidence is reproducible, unstable, baseline-confounded, disease-context-specific, or underpowered.

## Manuscript framing

This project is best framed as a database/resource paper rather than as a high-impact biological discovery paper.

Recommended manuscript title:

**Molecular Concordance Index: a reproducible transcriptomic resource for evaluating ClinVar-annotated cardiomyopathy genes across human HCM and DCM cohorts**

## Repository structure

```text
results/resource_tables/
  MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv
  MCI_CARDIOMYOPATHY_RESOURCE_DATA_DICTIONARY_v0_2.csv

manuscript/resource_paper/
  00_RESOURCE_PAPER_POSITIONING.md
  01_REVISED_RESOURCE_ABSTRACT.md
  02_REVISED_CONTRIBUTIONS.md
  03_RESOURCE_MANUSCRIPT_OUTLINE.md

docs/resource_framing/
  CLAIM_BOUNDARY_STATEMENT.md
  RESOURCE_TABLE_USAGE_GUIDE.md

scripts/scrna_contextualization/
  Placeholder for future public single-cell RNA-seq contextualization module.

shiny_app/
  Browser files or future browser outputs.

SyntaxError: incomplete input (3596080716.py, line 43)

In [ ]:
# =========================
# STEP 4 FIXED: Create GitHub-facing resource README and documentation
# =========================

from pathlib import Path
import pandas as pd
import json, datetime

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")
assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

resource_table_path = REPO_DIR / "results" / "resource_tables" / "MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv"
assert resource_table_path.exists(), f"Resource table not found: {resource_table_path}"

resource = pd.read_csv(resource_table_path)

docs_dir = REPO_DIR / "docs" / "resource_framing"
manuscript_dir = REPO_DIR / "manuscript" / "resource_paper"
metadata_dir = REPO_DIR / "metadata"

docs_dir.mkdir(parents=True, exist_ok=True)
manuscript_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

n_genes = resource["gene_symbol"].nunique()
tier_counts = resource["MCI_tier"].value_counts(dropna=False).to_dict()
gtex_counts = resource["GTEx_low_confidence_flag"].value_counts(dropna=False).to_dict()
n_sigma = int(resource["sigma_disease_to_GTEx_ratio"].notna().sum())
n_dcm = int(resource["has_DCM_generalization_entry"].sum())

high_n = int(tier_counts.get("HIGH", 0))
moderate_n = int(tier_counts.get("MODERATE", 0))
unstable_n = int(tier_counts.get("UNSTABLE", 0))
insufficient_n = int(tier_counts.get("INSUFFICIENT_COHORT_COVERAGE", 0))

gtex_low_n = int(gtex_counts.get(True, 0))
gtex_not_low_n = int(gtex_counts.get(False, 0))

resource_readme = f'''
# Molecular Concordance Index Cardiomyopathy Resource

## Overview

This repository contains the Molecular Concordance Index, or MCI, cardiomyopathy resource. It is a reproducible transcriptomic evidence-auditing framework for ClinVar-annotated hypertrophic cardiomyopathy and dilated cardiomyopathy genes.

The goal of this resource is not to claim that every clinically annotated cardiomyopathy gene has reproducible bulk-transcriptomic evidence. Instead, the resource makes that question explicit and queryable.

MCI separates four evidence layers that are often conflated:

1. Clinical pathogenicity annotation.
2. Cross-cohort transcriptomic reproducibility.
3. Normal left-ventricle expression variability.
4. Independent validation or convergence evidence.

The central resource table allows users to inspect whether a gene is transcript-concordant, transcript-unstable, GTEx-low-confidence, DCM-generalizable, or insufficiently covered by current public data.

## Current resource version

Version: v0.2-resource-reframe-corrected-source

Primary master table:

results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv

Data dictionary:

results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_DATA_DICTIONARY_v0_2.csv

## Current resource content

The current HCM-primary resource contains:

- {n_genes} ClinVar-annotated HCM genes
- {high_n} HIGH MCI genes
- {moderate_n} MODERATE MCI genes
- {unstable_n} UNSTABLE MCI genes
- {insufficient_n} genes with insufficient cohort coverage
- {n_sigma} genes with disease-to-GTEx variability ratios
- {gtex_low_n} genes flagged as GTEx-low-confidence
- {gtex_not_low_n} genes not flagged as GTEx-low-confidence
- {n_dcm} genes with attached DCM generalization entries

## How to interpret MCI

MCI is a composite transcriptomic concordance score:

MCI_g = 0.40 * D_g + 0.35 * S_g + 0.25 * R_g

where:

- D_g = direction agreement across cohorts
- S_g = effect-size consistency across cohorts
- R_g = statistical reproducibility across cohorts

Tier labels:

- HIGH: MCI >= 0.70
- MODERATE: 0.45 <= MCI < 0.70
- UNSTABLE: MCI < 0.45
- INSUFFICIENT_COHORT_COVERAGE: fewer than two eligible cohorts

## GTEx baseline interpretation

The GTEx layer compares disease-associated variability against normal human left-ventricle expression variability.

ratio = sigma_disease / sigma_GTEx

If the ratio is less than or equal to 1.0, the gene is flagged as GTEx-low-confidence. This does not mean the gene is biologically irrelevant. It means the disease-associated bulk-transcriptomic variability does not exceed normal GTEx left-ventricle baseline variability in the current data.

This distinction is important because a gene may be clinically pathogenic and transcript-concordant while still requiring orthogonal support from genotype-stratified cohorts, single-cell analysis, proteomics, or functional validation.

## Main claim boundary

This resource does not claim that MCI proves causality, clinical actionability, or therapeutic validity.

The resource is designed for evidence auditing and target-prioritization support. It should be used to identify genes whose transcript-level evidence is reproducible, unstable, baseline-confounded, disease-context-specific, or underpowered.

## Manuscript framing

This project is best framed as a database/resource paper rather than as a high-impact biological discovery paper.

Recommended manuscript title:

Molecular Concordance Index: a reproducible transcriptomic resource for evaluating ClinVar-annotated cardiomyopathy genes across human HCM and DCM cohorts

## Repository structure

results/resource_tables/
  MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv
  MCI_CARDIOMYOPATHY_RESOURCE_DATA_DICTIONARY_v0_2.csv

manuscript/resource_paper/
  00_RESOURCE_PAPER_POSITIONING.md
  01_REVISED_RESOURCE_ABSTRACT.md
  02_REVISED_CONTRIBUTIONS.md
  03_RESOURCE_MANUSCRIPT_OUTLINE.md

docs/resource_framing/
  CLAIM_BOUNDARY_STATEMENT.md
  RESOURCE_TABLE_USAGE_GUIDE.md
  NON_SIGNIFICANT_RESULTS_FRAMING.md

scripts/scrna_contextualization/
  Placeholder for future public single-cell RNA-seq contextualization module.

shiny_app/
  Browser files or future browser outputs.

## Suggested citation language

This repository provides a reproducible MCI resource for evaluating transcriptomic concordance of ClinVar-annotated cardiomyopathy genes across public human heart datasets. Users should cite the associated manuscript or preprint once available.

## Status

Active resource-paper reframing in progress.
'''

(REPO_DIR / "README_RESOURCE.md").write_text(resource_readme)

usage_guide = '''
# Resource Table Usage Guide

## Master table

File:

results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv

This is the main browser-ready table for the MCI cardiomyopathy resource.

## Recommended user workflow

### 1. Search by gene

Use gene_symbol to find a ClinVar-annotated cardiomyopathy gene.

### 2. Check primary MCI

Use:

- MCI
- MCI_tier
- MCI_CI95_lower
- MCI_CI95_upper
- bootstrap_majority_tier

These fields show whether the gene has reproducible transcript-level evidence across current HCM cohorts.

### 3. Check GTEx baseline status

Use:

- sigma_disease
- sigma_GTEx
- sigma_disease_to_GTEx_ratio
- GTEx_low_confidence_flag
- GTEx_adjustment_status

These fields show whether disease-associated variability exceeds normal left-ventricle expression variability.

### 4. Check DCM generalization

Use:

- has_DCM_generalization_entry
- DCM_MCI_if_available
- DCM_tier_if_available

These fields show whether a comparable DCM evidence layer was available.

### 5. Read plain-language interpretation

Use:

- resource_interpretation

This field summarizes how to interpret the row for target-evidence review.

## Important interpretation examples

### HIGH MCI plus GTEx-low-confidence

This means the gene is reproducible across HCM cohorts, but the magnitude of disease-associated variability does not exceed GTEx normal left-ventricle baseline variability. The gene should be treated as transcript-concordant but not as an unqualified transcript-level target.

### UNSTABLE MCI

This means the gene has inconsistent transcript-level evidence across current HCM cohorts. It may still be clinically important, especially if its mechanism is protein-level, structural, splicing-level, or cell-type-specific.

### DCM MCI missing

This does not mean the gene has no DCM relevance. It means DCM generalization evidence was not available or not attached in the current resource version.

## Suggested manuscript wording

The MCI resource should be described as an evidence-auditing layer that distinguishes clinical pathogenicity from transcriptomic reproducibility and normal-tissue variability. It is not a clinical decision tool and should not be interpreted as evidence of therapeutic validity without additional validation.
'''

(docs_dir / "RESOURCE_TABLE_USAGE_GUIDE.md").write_text(usage_guide)

nonsig_doc = '''
# How to Frame Non-Significant Results in the Resource Paper

## Core principle

In the resource-paper framing, non-significant validation modules are not treated as manuscript failure. They are reported as evidence layers that define what the current public data can and cannot support.

## H4 mechanism-stratified analysis

Original risk:
The HCM sarcomeric versus non-sarcomeric comparison was directionally consistent but not statistically significant.

Resource-paper framing:
This becomes a pre-specified demonstration analysis showing how users can test biological mechanism strata within the MCI resource. The result is reported as directional but underpowered, not as the central discovery claim.

## DCM generalization

Original risk:
The expected DCM pattern was not confirmed.

Resource-paper framing:
This becomes evidence that transcriptomic concordance may be disease-context-specific. The resource is valuable because it prevents automatic transfer of HCM evidence into DCM without testing.

## Held-out validation

Original risk:
The held-out validation trend did not reach statistical significance.

Resource-paper framing:
This becomes an honest replication module showing that the resource can evaluate whether high-MCI genes generalize to a quarantined cohort, while acknowledging limited power.

## GWAS convergence

Original risk:
GWAS convergence was directionally supportive but non-significant.

Resource-paper framing:
This supports the conclusion that transcriptomic concordance and inherited genetic association are non-equivalent evidence layers. The resource is useful because it exposes both alignment and mismatch.

## Recommended sentence

The validation modules were directionally informative but did not reach statistical significance, consistent with the role of MCI as a transparent evidence-auditing resource rather than a binary discovery test.
'''

(docs_dir / "NON_SIGNIFICANT_RESULTS_FRAMING.md").write_text(nonsig_doc)

summary = {
    "step": "STEP_4_RESOURCE_README_AND_DOCUMENTATION_FIXED",
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "resource_version": "v0.2-resource-reframe-corrected-source",
    "master_table": "results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv",
    "n_genes": int(n_genes),
    "tier_counts": {str(k): int(v) for k, v in tier_counts.items()},
    "gtex_low_confidence_counts": {str(k): int(v) for k, v in gtex_counts.items()},
    "n_with_sigma_ratio": n_sigma,
    "n_with_dcm_entry": n_dcm,
    "files_created": [
        "README_RESOURCE.md",
        "docs/resource_framing/RESOURCE_TABLE_USAGE_GUIDE.md",
        "docs/resource_framing/NON_SIGNIFICANT_RESULTS_FRAMING.md"
    ]
}

summary_path = metadata_dir / "step4_resource_readme_documentation_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("STEP 4 FIXED COMPLETE.")
print("Created:")
print(" -", REPO_DIR / "README_RESOURCE.md")
print(" -", docs_dir / "RESOURCE_TABLE_USAGE_GUIDE.md")
print(" -", docs_dir / "NON_SIGNIFICANT_RESULTS_FRAMING.md")
print(" -", summary_path)

print("\nResource summary:")
print(json.dumps(summary, indent=2))

STEP 4 FIXED COMPLETE.
Created:
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/README_RESOURCE.md
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/docs/resource_framing/RESOURCE_TABLE_USAGE_GUIDE.md
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/docs/resource_framing/NON_SIGNIFICANT_RESULTS_FRAMING.md
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/metadata/step4_resource_readme_documentation_summary.json

Resource summary:
{
  "step": "STEP_4_RESOURCE_README_AND_DOCUMENTATION_FIXED",
  "timestamp_utc": "2026-07-09T21:43:13.781057+00:00",
  "resource_version": "v0.2-resource-reframe-corrected-source",
  "master_table": "results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv",
  "n_genes": 49,
  "tier_counts": {
    "UNSTABLE": 21,
    "HIGH": 18,
    "MODERATE": 8,
    "INSUFFICIENT_COHORT_COVERAGE": 2
  },
  "gtex_low_confidence_counts": {
    "True": 46,
    "False": 3
  },
  "n_wi

In [ ]:
# =========================
# STEP 4 FIXED: Create GitHub-facing resource README and documentation
# =========================

from pathlib import Path
import pandas as pd
import json, datetime

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")
assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

resource_table_path = REPO_DIR / "results" / "resource_tables" / "MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv"
assert resource_table_path.exists(), f"Resource table not found: {resource_table_path}"

resource = pd.read_csv(resource_table_path)

docs_dir = REPO_DIR / "docs" / "resource_framing"
manuscript_dir = REPO_DIR / "manuscript" / "resource_paper"
metadata_dir = REPO_DIR / "metadata"

docs_dir.mkdir(parents=True, exist_ok=True)
manuscript_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

n_genes = resource["gene_symbol"].nunique()
tier_counts = resource["MCI_tier"].value_counts(dropna=False).to_dict()
gtex_counts = resource["GTEx_low_confidence_flag"].value_counts(dropna=False).to_dict()
n_sigma = int(resource["sigma_disease_to_GTEx_ratio"].notna().sum())
n_dcm = int(resource["has_DCM_generalization_entry"].sum())

high_n = int(tier_counts.get("HIGH", 0))
moderate_n = int(tier_counts.get("MODERATE", 0))
unstable_n = int(tier_counts.get("UNSTABLE", 0))
insufficient_n = int(tier_counts.get("INSUFFICIENT_COHORT_COVERAGE", 0))

gtex_low_n = int(gtex_counts.get(True, 0))
gtex_not_low_n = int(gtex_counts.get(False, 0))

resource_readme = f'''
# Molecular Concordance Index Cardiomyopathy Resource

## Overview

This repository contains the Molecular Concordance Index, or MCI, cardiomyopathy resource. It is a reproducible transcriptomic evidence-auditing framework for ClinVar-annotated hypertrophic cardiomyopathy and dilated cardiomyopathy genes.

The goal of this resource is not to claim that every clinically annotated cardiomyopathy gene has reproducible bulk-transcriptomic evidence. Instead, the resource makes that question explicit and queryable.

MCI separates four evidence layers that are often conflated:

1. Clinical pathogenicity annotation.
2. Cross-cohort transcriptomic reproducibility.
3. Normal left-ventricle expression variability.
4. Independent validation or convergence evidence.

The central resource table allows users to inspect whether a gene is transcript-concordant, transcript-unstable, GTEx-low-confidence, DCM-generalizable, or insufficiently covered by current public data.

## Current resource version

Version: v0.2-resource-reframe-corrected-source

Primary master table:

results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv

Data dictionary:

results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_DATA_DICTIONARY_v0_2.csv

## Current resource content

The current HCM-primary resource contains:

- {n_genes} ClinVar-annotated HCM genes
- {high_n} HIGH MCI genes
- {moderate_n} MODERATE MCI genes
- {unstable_n} UNSTABLE MCI genes
- {insufficient_n} genes with insufficient cohort coverage
- {n_sigma} genes with disease-to-GTEx variability ratios
- {gtex_low_n} genes flagged as GTEx-low-confidence
- {gtex_not_low_n} genes not flagged as GTEx-low-confidence
- {n_dcm} genes with attached DCM generalization entries

## How to interpret MCI

MCI is a composite transcriptomic concordance score:

MCI_g = 0.40 * D_g + 0.35 * S_g + 0.25 * R_g

where:

- D_g = direction agreement across cohorts
- S_g = effect-size consistency across cohorts
- R_g = statistical reproducibility across cohorts

Tier labels:

- HIGH: MCI >= 0.70
- MODERATE: 0.45 <= MCI < 0.70
- UNSTABLE: MCI < 0.45
- INSUFFICIENT_COHORT_COVERAGE: fewer than two eligible cohorts

## GTEx baseline interpretation

The GTEx layer compares disease-associated variability against normal human left-ventricle expression variability.

ratio = sigma_disease / sigma_GTEx

If the ratio is less than or equal to 1.0, the gene is flagged as GTEx-low-confidence. This does not mean the gene is biologically irrelevant. It means the disease-associated bulk-transcriptomic variability does not exceed normal GTEx left-ventricle baseline variability in the current data.

This distinction is important because a gene may be clinically pathogenic and transcript-concordant while still requiring orthogonal support from genotype-stratified cohorts, single-cell analysis, proteomics, or functional validation.

## Main claim boundary

This resource does not claim that MCI proves causality, clinical actionability, or therapeutic validity.

The resource is designed for evidence auditing and target-prioritization support. It should be used to identify genes whose transcript-level evidence is reproducible, unstable, baseline-confounded, disease-context-specific, or underpowered.

## Manuscript framing

This project is best framed as a database/resource paper rather than as a high-impact biological discovery paper.

Recommended manuscript title:

Molecular Concordance Index: a reproducible transcriptomic resource for evaluating ClinVar-annotated cardiomyopathy genes across human HCM and DCM cohorts

## Repository structure

results/resource_tables/
  MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv
  MCI_CARDIOMYOPATHY_RESOURCE_DATA_DICTIONARY_v0_2.csv

manuscript/resource_paper/
  00_RESOURCE_PAPER_POSITIONING.md
  01_REVISED_RESOURCE_ABSTRACT.md
  02_REVISED_CONTRIBUTIONS.md
  03_RESOURCE_MANUSCRIPT_OUTLINE.md

docs/resource_framing/
  CLAIM_BOUNDARY_STATEMENT.md
  RESOURCE_TABLE_USAGE_GUIDE.md
  NON_SIGNIFICANT_RESULTS_FRAMING.md

scripts/scrna_contextualization/
  Placeholder for future public single-cell RNA-seq contextualization module.

shiny_app/
  Browser files or future browser outputs.

## Suggested citation language

This repository provides a reproducible MCI resource for evaluating transcriptomic concordance of ClinVar-annotated cardiomyopathy genes across public human heart datasets. Users should cite the associated manuscript or preprint once available.

## Status

Active resource-paper reframing in progress.
'''

(REPO_DIR / "README_RESOURCE.md").write_text(resource_readme)

usage_guide = '''
# Resource Table Usage Guide

## Master table

File:

results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv

This is the main browser-ready table for the MCI cardiomyopathy resource.

## Recommended user workflow

### 1. Search by gene

Use gene_symbol to find a ClinVar-annotated cardiomyopathy gene.

### 2. Check primary MCI

Use:

- MCI
- MCI_tier
- MCI_CI95_lower
- MCI_CI95_upper
- bootstrap_majority_tier

These fields show whether the gene has reproducible transcript-level evidence across current HCM cohorts.

### 3. Check GTEx baseline status

Use:

- sigma_disease
- sigma_GTEx
- sigma_disease_to_GTEx_ratio
- GTEx_low_confidence_flag
- GTEx_adjustment_status

These fields show whether disease-associated variability exceeds normal left-ventricle expression variability.

### 4. Check DCM generalization

Use:

- has_DCM_generalization_entry
- DCM_MCI_if_available
- DCM_tier_if_available

These fields show whether a comparable DCM evidence layer was available.

### 5. Read plain-language interpretation

Use:

- resource_interpretation

This field summarizes how to interpret the row for target-evidence review.

## Important interpretation examples

### HIGH MCI plus GTEx-low-confidence

This means the gene is reproducible across HCM cohorts, but the magnitude of disease-associated variability does not exceed GTEx normal left-ventricle baseline variability. The gene should be treated as transcript-concordant but not as an unqualified transcript-level target.

### UNSTABLE MCI

This means the gene has inconsistent transcript-level evidence across current HCM cohorts. It may still be clinically important, especially if its mechanism is protein-level, structural, splicing-level, or cell-type-specific.

### DCM MCI missing

This does not mean the gene has no DCM relevance. It means DCM generalization evidence was not available or not attached in the current resource version.

## Suggested manuscript wording

The MCI resource should be described as an evidence-auditing layer that distinguishes clinical pathogenicity from transcriptomic reproducibility and normal-tissue variability. It is not a clinical decision tool and should not be interpreted as evidence of therapeutic validity without additional validation.
'''

(docs_dir / "RESOURCE_TABLE_USAGE_GUIDE.md").write_text(usage_guide)

nonsig_doc = '''
# How to Frame Non-Significant Results in the Resource Paper

## Core principle

In the resource-paper framing, non-significant validation modules are not treated as manuscript failure. They are reported as evidence layers that define what the current public data can and cannot support.

## H4 mechanism-stratified analysis

Original risk:
The HCM sarcomeric versus non-sarcomeric comparison was directionally consistent but not statistically significant.

Resource-paper framing:
This becomes a pre-specified demonstration analysis showing how users can test biological mechanism strata within the MCI resource. The result is reported as directional but underpowered, not as the central discovery claim.

## DCM generalization

Original risk:
The expected DCM pattern was not confirmed.

Resource-paper framing:
This becomes evidence that transcriptomic concordance may be disease-context-specific. The resource is valuable because it prevents automatic transfer of HCM evidence into DCM without testing.

## Held-out validation

Original risk:
The held-out validation trend did not reach statistical significance.

Resource-paper framing:
This becomes an honest replication module showing that the resource can evaluate whether high-MCI genes generalize to a quarantined cohort, while acknowledging limited power.

## GWAS convergence

Original risk:
GWAS convergence was directionally supportive but non-significant.

Resource-paper framing:
This supports the conclusion that transcriptomic concordance and inherited genetic association are non-equivalent evidence layers. The resource is useful because it exposes both alignment and mismatch.

## Recommended sentence

The validation modules were directionally informative but did not reach statistical significance, consistent with the role of MCI as a transparent evidence-auditing resource rather than a binary discovery test.
'''

(docs_dir / "NON_SIGNIFICANT_RESULTS_FRAMING.md").write_text(nonsig_doc)

summary = {
    "step": "STEP_4_RESOURCE_README_AND_DOCUMENTATION_FIXED",
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "resource_version": "v0.2-resource-reframe-corrected-source",
    "master_table": "results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv",
    "n_genes": int(n_genes),
    "tier_counts": {str(k): int(v) for k, v in tier_counts.items()},
    "gtex_low_confidence_counts": {str(k): int(v) for k, v in gtex_counts.items()},
    "n_with_sigma_ratio": n_sigma,
    "n_with_dcm_entry": n_dcm,
    "files_created": [
        "README_RESOURCE.md",
        "docs/resource_framing/RESOURCE_TABLE_USAGE_GUIDE.md",
        "docs/resource_framing/NON_SIGNIFICANT_RESULTS_FRAMING.md"
    ]
}

summary_path = metadata_dir / "step4_resource_readme_documentation_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("STEP 4 FIXED COMPLETE.")
print("Created:")
print(" -", REPO_DIR / "README_RESOURCE.md")
print(" -", docs_dir / "RESOURCE_TABLE_USAGE_GUIDE.md")
print(" -", docs_dir / "NON_SIGNIFICANT_RESULTS_FRAMING.md")
print(" -", summary_path)

print("\nResource summary:")
print(json.dumps(summary, indent=2))

STEP 4 FIXED COMPLETE.
Created:
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/README_RESOURCE.md
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/docs/resource_framing/RESOURCE_TABLE_USAGE_GUIDE.md
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/docs/resource_framing/NON_SIGNIFICANT_RESULTS_FRAMING.md
 - /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/metadata/step4_resource_readme_documentation_summary.json

Resource summary:
{
  "step": "STEP_4_RESOURCE_README_AND_DOCUMENTATION_FIXED",
  "timestamp_utc": "2026-07-09T21:44:14.101259+00:00",
  "resource_version": "v0.2-resource-reframe-corrected-source",
  "master_table": "results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv",
  "n_genes": 49,
  "tier_counts": {
    "UNSTABLE": 21,
    "HIGH": 18,
    "MODERATE": 8,
    "INSUFFICIENT_COHORT_COVERAGE": 2
  },
  "gtex_low_confidence_counts": {
    "True": 46,
    "False": 3
  },
  "n_wi

In [ ]:
# =========================
# STEP 5: Create scRNA-seq contextualization module scaffold
# =========================

from pathlib import Path
import pandas as pd
import json, datetime

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")
assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

scrna_dir = REPO_DIR / "scripts" / "scrna_contextualization"
docs_dir = REPO_DIR / "docs" / "resource_framing"
metadata_dir = REPO_DIR / "metadata"
results_dir = REPO_DIR / "results" / "resource_tables"

scrna_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

resource_table_path = results_dir / "MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv"
assert resource_table_path.exists(), f"Resource table missing: {resource_table_path}"

resource = pd.read_csv(resource_table_path)
target_genes = sorted(resource["gene_symbol"].dropna().astype(str).str.upper().unique())

# -------------------------
# 1. Create scRNA target gene list
# -------------------------

scrna_gene_list = pd.DataFrame({
    "gene_symbol": target_genes,
    "included_in_MCI_resource_v0_2": True,
    "planned_scrna_fields": "cardiomyocyte_expression,fibroblast_expression,endothelial_expression,immune_expression,dominant_cell_type,cell_type_specificity_score"
})

gene_list_path = results_dir / "MCI_scRNA_contextualization_target_gene_list_v0_1.csv"
scrna_gene_list.to_csv(gene_list_path, index=False)

# -------------------------
# 2. Create scRNA module README
# -------------------------

scrna_readme = """
# scRNA-seq Contextualization Module

## Purpose

This module is designed to extend the MCI cardiomyopathy resource from bulk-tissue transcriptomic concordance into cell-type contextualization using public human cardiac single-cell or single-nucleus RNA-seq datasets.

The goal is not to replace the bulk MCI score. The goal is to explain whether bulk-transcript instability may reflect cell-type dilution, cell-type specificity, or disease-cell-composition effects.

## Scientific motivation

A gene can be clinically important but appear unstable in bulk RNA-seq for several reasons:

1. The gene may be expressed mainly in cardiomyocytes but diluted by fibroblast, endothelial, smooth muscle, or immune-cell composition.
2. The disease signal may exist only in a specific cell type.
3. The gene may act through protein-level or splicing-level mechanisms rather than bulk abundance.
4. Public bulk cohorts may differ in cellular composition, disease stage, tissue region, or genotype mix.

A single-cell contextualization layer can therefore strengthen the resource-paper framing by showing how MCI genes behave across cardiac cell types.

## Planned input

A public human HCM or DCM single-cell/single-nucleus RNA-seq dataset with:

- expression matrix or AnnData object,
- cell barcode metadata,
- cell-type labels,
- disease status if available,
- human gene symbols.

## Planned output

The planned output is a gene-level annotation table:

- gene_symbol
- cardiomyocyte_mean_expression
- fibroblast_mean_expression
- endothelial_mean_expression
- immune_mean_expression
- smooth_muscle_mean_expression
- dominant_cell_type
- cell_type_specificity_score
- detected_in_cardiomyocytes
- detected_in_fibroblasts
- MCI_tier
- GTEx_low_confidence_flag
- interpretation

## Manuscript role

This module should be presented as a cell-type contextualization layer for the database/resource paper. It should not be overclaimed as wet-lab validation.

Recommended sentence:

To support cell-type interpretation of bulk concordance patterns, we added a planned scRNA-seq contextualization module that annotates MCI genes by cardiac cell-type expression and identifies genes whose apparent bulk instability may reflect cell-type specificity or cellular composition effects.

## Current status

Scaffold created. Dataset selection and execution remain pending.
"""

(scrna_dir / "README.md").write_text(scrna_readme)

# -------------------------
# 3. Create executable Python script scaffold
# -------------------------

script_text = r'''
"""
scRNA-seq contextualization module for the MCI cardiomyopathy resource.

This script is a scaffold. It is designed for public human cardiac
single-cell or single-nucleus RNA-seq datasets loaded as AnnData `.h5ad`
files.

Expected input:
    1. AnnData file with genes in adata.var_names
    2. Cell metadata column containing cell-type labels
    3. MCI resource master table CSV

Expected output:
    Gene-level cell-type contextualization table.

Example:
    python scripts/scrna_contextualization/run_scrna_contextualization.py \
        --h5ad data/external/scrna/example_human_heart.h5ad \
        --mci_table results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv \
        --celltype_col cell_type \
        --output results/resource_tables/MCI_scRNA_celltype_context_v0_1.csv
"""

import argparse
from pathlib import Path
import pandas as pd
import numpy as np


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--h5ad", required=True, help="Path to AnnData .h5ad file.")
    parser.add_argument("--mci_table", required=True, help="Path to MCI resource master table.")
    parser.add_argument("--celltype_col", required=True, help="Cell-type column in adata.obs.")
    parser.add_argument("--output", required=True, help="Output CSV path.")
    return parser.parse_args()


def main():
    args = parse_args()

    try:
        import scanpy as sc
    except ImportError as exc:
        raise ImportError(
            "scanpy is required for this module. Install with: pip install scanpy anndata"
        ) from exc

    h5ad_path = Path(args.h5ad)
    mci_path = Path(args.mci_table)
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    adata = sc.read_h5ad(h5ad_path)
    mci = pd.read_csv(mci_path)

    if args.celltype_col not in adata.obs.columns:
        raise ValueError(f"Cell-type column not found in adata.obs: {args.celltype_col}")

    if "gene_symbol" not in mci.columns:
        raise ValueError("MCI table must contain gene_symbol column.")

    # Standardize gene symbols
    mci["gene_symbol"] = mci["gene_symbol"].astype(str).str.upper().str.strip()
    adata.var_names = adata.var_names.astype(str).str.upper().str.strip()

    target_genes = sorted(set(mci["gene_symbol"]) & set(adata.var_names))
    missing_genes = sorted(set(mci["gene_symbol"]) - set(adata.var_names))

    if len(target_genes) == 0:
        raise ValueError("No MCI target genes found in AnnData var_names.")

    # Subset target genes
    adata_sub = adata[:, target_genes].copy()

    # Convert to dense only for small target-gene subset
    X = adata_sub.X
    if hasattr(X, "toarray"):
        X = X.toarray()

    expr = pd.DataFrame(X, columns=target_genes, index=adata_sub.obs_names)
    expr[args.celltype_col] = adata_sub.obs[args.celltype_col].astype(str).values

    # Mean expression by cell type
    means = expr.groupby(args.celltype_col)[target_genes].mean().T
    means.index.name = "gene_symbol"
    means = means.reset_index()

    # Long format for specificity computation
    long_df = means.melt(id_vars="gene_symbol", var_name="cell_type", value_name="mean_expression")

    # Dominant cell type and specificity
    dominant = (
        long_df.sort_values(["gene_symbol", "mean_expression"], ascending=[True, False])
        .groupby("gene_symbol")
        .head(1)
        .rename(columns={"cell_type": "dominant_cell_type", "mean_expression": "dominant_celltype_mean_expression"})
    )

    total_expr = long_df.groupby("gene_symbol")["mean_expression"].sum().reset_index()
    total_expr = total_expr.rename(columns={"mean_expression": "sum_mean_expression_across_celltypes"})

    dominant = dominant.merge(total_expr, on="gene_symbol", how="left")
    dominant["cell_type_specificity_score"] = (
        dominant["dominant_celltype_mean_expression"] /
        dominant["sum_mean_expression_across_celltypes"].replace(0, np.nan)
    )

    # Wide cell-type table with safe column names
    means_wide = means.copy()
    safe_cols = []
    for col in means_wide.columns:
        if col == "gene_symbol":
            safe_cols.append(col)
        else:
            safe = str(col).lower().replace(" ", "_").replace("/", "_").replace("-", "_")
            safe_cols.append(f"mean_expression_{safe}")
    means_wide.columns = safe_cols

    out = mci.merge(means_wide, on="gene_symbol", how="left")
    out = out.merge(
        dominant[[
            "gene_symbol",
            "dominant_cell_type",
            "dominant_celltype_mean_expression",
            "cell_type_specificity_score"
        ]],
        on="gene_symbol",
        how="left"
    )

    out["matched_in_scrna_dataset"] = out["gene_symbol"].isin(target_genes)
    out["scrna_missing_reason"] = np.where(
        out["matched_in_scrna_dataset"],
        "",
        "gene_symbol_not_found_in_scrna_var_names"
    )

    out.to_csv(output_path, index=False)

    summary = {
        "h5ad": str(h5ad_path),
        "mci_table": str(mci_path),
        "output": str(output_path),
        "celltype_col": args.celltype_col,
        "n_mci_genes": int(mci["gene_symbol"].nunique()),
        "n_matched_genes": int(len(target_genes)),
        "n_missing_genes": int(len(missing_genes)),
        "missing_genes": missing_genes[:100],
        "n_cells": int(adata.n_obs),
        "n_genes_in_scrna": int(adata.n_vars),
        "cell_type_counts": adata.obs[args.celltype_col].astype(str).value_counts().to_dict()
    }

    summary_path = output_path.with_suffix(".summary.json")
    summary_path.write_text(pd.Series(summary).to_json(indent=2))

    print("scRNA contextualization complete.")
    print("Output:", output_path)
    print("Summary:", summary_path)


if __name__ == "__main__":
    main()
'''

script_path = scrna_dir / "run_scrna_contextualization.py"
script_path.write_text(script_text)

# -------------------------
# 4. Create manuscript module text
# -------------------------

manuscript_text = """
# Optional Manuscript Text: scRNA-seq Contextualization Module

Bulk heart transcriptomic cohorts are affected by cellular composition, tissue-region sampling, disease stage, and genotype heterogeneity. Therefore, unstable bulk MCI values should not automatically be interpreted as absence of biological relevance. To support future resource expansion, we created a single-cell RNA-seq contextualization module that maps MCI genes onto public human cardiac single-cell or single-nucleus RNA-seq datasets.

The module is designed to annotate each MCI gene by cardiac cell-type expression, including cardiomyocyte, fibroblast, endothelial, smooth-muscle, and immune-cell expression where those labels are available. For each gene, the planned output reports mean expression by cell type, dominant cell type, cell-type specificity score, MCI tier, GTEx confidence status, and a row-level interpretation.

This layer is intended to help distinguish truly unstable bulk transcriptomic evidence from signals that may be diluted by cell-type composition. For example, a sarcomeric gene with low bulk MCI but strong cardiomyocyte-specific expression may require cardiomyocyte-resolved analysis, proteomics, or genotype-stratified validation rather than being deprioritized solely on the basis of bulk RNA-seq instability.
"""

(manuscript_dir / "04_OPTIONAL_scRNA_CONTEXTUALIZATION_MODULE_TEXT.md").write_text(manuscript_text)

# -------------------------
# 5. Save metadata
# -------------------------

summary = {
    "step": "STEP_5_scRNA_CONTEXTUALIZATION_MODULE_SCAFFOLD",
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "n_target_genes": int(len(target_genes)),
    "files_created": [
        str(gene_list_path.relative_to(REPO_DIR)),
        str((scrna_dir / "README.md").relative_to(REPO_DIR)),
        str(script_path.relative_to(REPO_DIR)),
        str((manuscript_dir / "04_OPTIONAL_scRNA_CONTEXTUALIZATION_MODULE_TEXT.md").relative_to(REPO_DIR))
    ],
    "status": "Scaffold created. Dataset selection and execution pending."
}

summary_path = metadata_dir / "step5_scrna_contextualization_scaffold_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("STEP 5 COMPLETE.")
print("Created:")
for f in summary["files_created"]:
    print(" -", f)

print("\nTarget genes:", len(target_genes))
print("Summary:", summary_path)

STEP 5 COMPLETE.
Created:
 - results/resource_tables/MCI_scRNA_contextualization_target_gene_list_v0_1.csv
 - scripts/scrna_contextualization/README.md
 - scripts/scrna_contextualization/run_scrna_contextualization.py
 - manuscript/resource_paper/04_OPTIONAL_scRNA_CONTEXTUALIZATION_MODULE_TEXT.md

Target genes: 49
Summary: /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/metadata/step5_scrna_contextualization_scaffold_summary.json


In [ ]:
# =========================
# STEP 6: Generate resource-paper manuscript skeleton from corrected resource table
# =========================

from pathlib import Path
import pandas as pd
import json, datetime

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")
assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

resource_table_path = REPO_DIR / "results" / "resource_tables" / "MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv"
assert resource_table_path.exists(), f"Resource table not found: {resource_table_path}"

resource = pd.read_csv(resource_table_path)

manuscript_dir = REPO_DIR / "manuscript" / "resource_paper"
metadata_dir = REPO_DIR / "metadata"
manuscript_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

# -------------------------
# Summary statistics
# -------------------------

n_genes = resource["gene_symbol"].nunique()

tier_counts = resource["MCI_tier"].value_counts(dropna=False).to_dict()
high_n = int(tier_counts.get("HIGH", 0))
moderate_n = int(tier_counts.get("MODERATE", 0))
unstable_n = int(tier_counts.get("UNSTABLE", 0))
insufficient_n = int(tier_counts.get("INSUFFICIENT_COHORT_COVERAGE", 0))

gtex_counts = resource["GTEx_low_confidence_flag"].value_counts(dropna=False).to_dict()
gtex_low_n = int(gtex_counts.get(True, 0))
gtex_not_low_n = int(gtex_counts.get(False, 0))

n_sigma = int(resource["sigma_disease_to_GTEx_ratio"].notna().sum())
n_dcm = int(resource["has_DCM_generalization_entry"].sum())

median_mci = float(resource["MCI"].median())
mean_mci = float(resource["MCI"].mean())

top_high = (
    resource.sort_values("MCI", ascending=False)
    .loc[:, ["gene_symbol", "MCI", "Adj_MCI", "MCI_tier", "GTEx_low_confidence_flag"]]
    .head(10)
)

unstable_genes = (
    resource[resource["MCI_tier"] == "UNSTABLE"]
    .sort_values("MCI", ascending=True)
    .loc[:, ["gene_symbol", "MCI", "MCI_tier", "GTEx_low_confidence_flag"]]
    .head(10)
)

top_high_md = top_high.to_markdown(index=False)
unstable_md = unstable_genes.to_markdown(index=False)

# -------------------------
# Manuscript skeleton text
# -------------------------

manuscript = f"""
# Molecular Concordance Index: a reproducible transcriptomic resource for evaluating ClinVar-annotated cardiomyopathy genes across human HCM and DCM cohorts

Sanghati Basu and collaborators

## Abstract

Clinical pathogenicity annotation is essential for cardiomyopathy genetics, but it does not establish whether a gene shows reproducible transcript-level perturbation across independent human disease cohorts. This creates a translational informatics gap: genes may be clinically important while remaining unstable, cohort-specific, or indistinguishable from normal-tissue variability at the bulk-transcriptome level. We present the Molecular Concordance Index, or MCI, a reproducible resource and scoring framework for evaluating transcriptomic concordance of ClinVar-annotated hypertrophic cardiomyopathy and dilated cardiomyopathy genes across public human heart datasets.

MCI integrates three evidence layers: direction agreement across cohorts, effect-size consistency, and statistical reproducibility. We applied the framework to public HCM cohorts, extended the analysis to DCM generalization datasets, added 1,000-iteration cohort-level bootstrap confidence intervals, and benchmarked disease-associated variability against GTEx v8 Heart - Left Ventricle expression variability. The resulting resource assigns each gene a concordance score, bootstrap interval, tier classification, GTEx low-confidence flag, and per-cohort differential-expression evidence.

Across {n_genes} HCM genes, the resource identified heterogeneous transcript-level behavior: {high_n} genes were classified as high concordance, {moderate_n} as moderate, {unstable_n} as unstable, and {insufficient_n} as insufficient coverage. GTEx benchmarking showed that {gtex_low_n} of {n_genes} genes were flagged as GTEx-low-confidence, indicating that disease-associated variability often did not exceed normal left-ventricle baseline variability in the current bulk data. DCM generalization entries were available for {n_dcm} genes. Pre-specified mechanism-stratified, DCM generalization, held-out replication, and GWAS convergence analyses were directionally informative but did not provide definitive statistically significant discovery claims.

The MCI resource therefore provides a transparent evidence-auditing layer for cardiomyopathy target evaluation rather than a binary discovery test. It enables researchers to distinguish clinically annotated genes with reproducible transcriptomic support from genes whose disease signal is unstable, baseline-confounded, disease-context-specific, or underpowered. All resource tables, figures, and browser-ready outputs are designed for public reuse and extension, including future single-cell and genotype-stratified modules.

## Keywords

cardiomyopathy; ClinVar; transcriptomics; target validation; molecular concordance; GTEx; hypertrophic cardiomyopathy; dilated cardiomyopathy; database resource; biomedical informatics

## 1. Introduction

Inherited cardiomyopathies are genetically heterogeneous disorders in which clinical interpretation often relies on curated pathogenic and likely pathogenic variant annotations. ClinVar and related clinical-genomic resources are therefore essential for identifying genes implicated in hypertrophic cardiomyopathy and dilated cardiomyopathy. However, clinical pathogenicity annotation and transcriptomic reproducibility are not the same evidence layer. A gene may be clinically causal through protein structure, sarcomere mechanics, splicing, ion handling, or cell-type-specific mechanisms without showing reproducible bulk-transcript abundance change across independent disease cohorts.

This distinction matters for translational informatics. Public human heart transcriptomic datasets are increasingly used to prioritize biomarkers, nominate targets, and justify downstream validation. Yet there is no standard resource that asks whether ClinVar-annotated cardiomyopathy genes show reproducible direction, effect size, and statistical evidence across independent human heart cohorts. There is also limited separation between disease-cohort reproducibility and baseline variability in normal human left ventricle.

The Molecular Concordance Index resource addresses this gap. Instead of treating one differential-expression result as sufficient evidence, MCI evaluates whether a gene behaves consistently across cohorts. It combines direction agreement, effect-size consistency, and statistical reproducibility into a gene-level concordance score. The resource further adds bootstrap uncertainty, GTEx v8 Heart - Left Ventricle baseline benchmarking, DCM generalization, held-out validation, GWAS convergence, and browser-ready outputs.

The purpose of this manuscript is therefore not to claim a single definitive cardiomyopathy mechanism discovery. The purpose is to present a reusable evidence-auditing resource. The core claim is that clinical pathogenicity annotation, bulk transcriptomic concordance, normal-tissue expression variability, and independent genetic association are related but non-equivalent layers of evidence.

## 2. Results

### 2.1 Resource construction and cohort coverage

The MCI resource was constructed from public human heart transcriptomic cohorts, ClinVar-annotated cardiomyopathy gene sets, and GTEx v8 Heart - Left Ventricle baseline expression. The current HCM-primary resource contains {n_genes} ClinVar-annotated genes with MCI scoring or coverage status.

The primary resource table is:

`results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv`

A data dictionary is provided at:

`results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_DATA_DICTIONARY_v0_2.csv`

Each row includes gene symbol, disease group, pre-specified stratum where available, MCI, adjusted MCI, bootstrap interval, tier assignment, GTEx baseline fields, DCM generalization fields where available, and a plain-language interpretation field for browser display.

### 2.2 Primary HCM MCI resource

Across {n_genes} HCM genes, {high_n} were classified as HIGH, {moderate_n} as MODERATE, {unstable_n} as UNSTABLE, and {insufficient_n} as INSUFFICIENT_COHORT_COVERAGE. The median MCI was {median_mci:.3f}, and the mean MCI was {mean_mci:.3f}.

This distribution supports the main resource premise: clinically annotated cardiomyopathy genes do not behave uniformly at the bulk-transcriptomic level. Some genes show strong cross-cohort concordance, whereas others remain unstable or insufficiently covered.

#### Top high-concordance HCM genes in the current resource

{top_high_md}

#### Most unstable HCM genes in the current resource

{unstable_md}

### 2.3 Bootstrap uncertainty and tier stability

The resource includes cohort-level bootstrap uncertainty estimates. Bootstrap intervals are intentionally reported because the current public HCM evidence base contains a limited number of eligible cohorts. The intervals should therefore be interpreted as transparency measures rather than as high-powered uncertainty estimates.

The bootstrap layer supports a database-resource interpretation: users can inspect not only the point MCI tier, but also whether the tier is stable under cohort-level resampling and whether the confidence interval crosses tier thresholds.

### 2.4 GTEx baseline benchmarking

GTEx v8 Heart - Left Ventricle expression was used as a normal baseline benchmark. For each gene, disease-associated variability was compared against normal left-ventricle variability using:

`sigma_disease_to_GTEx_ratio = sigma_disease / sigma_GTEx`

In the current resource, {n_sigma} genes had available disease-to-GTEx variability ratios. {gtex_low_n} genes were flagged as GTEx-low-confidence, while {gtex_not_low_n} were not flagged as low-confidence.

A GTEx-low-confidence flag does not mean that a gene is biologically irrelevant. It means that, in the current bulk RNA-seq and microarray evidence, disease-associated variability does not exceed normal GTEx left-ventricle baseline variability. This is especially important for target interpretation because a gene can be clinically pathogenic and transcript-concordant while still requiring orthogonal support from single-cell, proteomic, genotype-stratified, or functional evidence.

### 2.5 Mechanism-stratified use case

A pre-specified mechanism-stratified analysis compared sarcomeric and non-sarcomeric HCM genes. This analysis should be interpreted as a demonstration of how biological strata can be evaluated within the MCI resource, not as the central discovery claim of the paper. The observed pattern was directionally informative but not statistically definitive.

This result is still useful in a resource-paper framework because it identifies where additional cohorts, genotype-stratified expression, proteomics, or single-cell analysis are needed before making strong mechanism-level claims.

### 2.6 DCM generalization module

The DCM layer evaluates whether concordance patterns observed in HCM extend into DCM datasets. In the current master table, {n_dcm} genes have attached DCM generalization entries.

The DCM module is not framed as proof that HCM and DCM transcriptomic concordance behave identically. Instead, it is a disease-context test. Where DCM patterns do not mirror HCM patterns, the resource highlights disease specificity rather than suppressing discordant evidence.

### 2.7 Held-out and GWAS validation modules

The resource includes held-out and GWAS convergence modules as independent evidence layers. These modules should be reported transparently even when they are directionally supportive but not statistically significant. Their value is to show whether MCI tiers align with unseen cohort behavior or inherited genetic association evidence.

In the resource-paper framing, these analyses are not required to produce a binary significant result. They define the current boundary of evidence and clarify where transcriptomic concordance, replication behavior, and GWAS signals agree or diverge.

### 2.8 Browser-ready output

The master table is designed for browser deployment. A gene-level browser should allow users to search a gene and retrieve:

- MCI and adjusted MCI
- MCI tier and bootstrap-majority tier
- bootstrap confidence interval
- GTEx low-confidence flag
- disease-to-GTEx variability ratio
- DCM generalization entry where available
- plain-language interpretation
- downloadable full table

This makes the project suitable for database/resource manuscript tracks.

### 2.9 Planned single-cell contextualization module

A scaffolded scRNA-seq contextualization module is included in:

`scripts/scrna_contextualization/`

This module is designed to annotate MCI genes by cardiac cell-type expression using public human single-cell or single-nucleus RNA-seq datasets. The goal is to determine whether unstable bulk MCI behavior may reflect cell-type dilution or cell-type specificity rather than absence of biological relevance.

## 3. Methods

### 3.1 Dataset acquisition

Public human heart transcriptomic datasets were used for HCM and DCM concordance scoring. The resource uses existing public data and does not require new sample collection.

### 3.2 ClinVar cardiomyopathy gene universe

ClinVar pathogenic and likely pathogenic cardiomyopathy annotations were used to define the gene universe. Genes were grouped into disease and mechanism strata where possible.

### 3.3 Differential-expression processing

Each cohort was analyzed independently to estimate disease-versus-control differential expression. The output for each cohort included gene symbol, log2 fold change, standard error where available, nominal p-value, and FDR-adjusted p-value.

### 3.4 Harmonization and batch-aware processing

Gene identifiers were standardized to HGNC-style gene symbols. Cohort-level outputs were harmonized into a common schema before MCI scoring. Batch-aware processing and harmonization steps were used where appropriate for expression preprocessing and cross-cohort comparability.

### 3.5 Molecular Concordance Index formula

For gene g, MCI was computed as:

`MCI_g = 0.40 * D_g + 0.35 * S_g + 0.25 * R_g`

where `D_g` is direction agreement, `S_g` is effect-size consistency, and `R_g` is statistical reproducibility.

### 3.6 Tier assignment

Genes were assigned to tiers using pre-specified thresholds:

- HIGH: MCI >= 0.70
- MODERATE: 0.45 <= MCI < 0.70
- UNSTABLE: MCI < 0.45
- INSUFFICIENT_COHORT_COVERAGE: fewer than two eligible cohorts

### 3.7 Bootstrap confidence intervals

Cohort-level bootstrap resampling was performed to estimate MCI uncertainty and tier stability. Bootstrap outputs include 95 percent confidence intervals, tier probabilities, and bootstrap-majority tier labels.

### 3.8 GTEx baseline adjustment

GTEx v8 Heart - Left Ventricle expression was used to calculate normal baseline variability. Disease-associated variability was compared against GTEx variability using `sigma_disease_to_GTEx_ratio`. Genes with ratio less than or equal to 1.0 were flagged as GTEx-low-confidence.

### 3.9 DCM generalization

DCM datasets were processed using the same concordance logic where available. DCM entries were attached to the master resource table as a disease-context generalization layer.

### 3.10 Held-out and GWAS validation

Held-out validation and GWAS convergence were implemented as independent validation modules. These results are interpreted as evidence layers rather than as required binary significance tests.

### 3.11 scRNA-seq contextualization scaffold

A planned scRNA-seq module was scaffolded to support future annotation of MCI genes by cardiac cell type. The module accepts public AnnData files and reports gene-level cell-type expression, dominant cell type, and cell-type specificity metrics.

## 4. Discussion

The MCI cardiomyopathy resource addresses a practical translational informatics problem: clinical pathogenicity annotation does not automatically establish reproducible bulk-transcriptomic evidence. By making transcriptomic concordance, GTEx baseline variability, bootstrap uncertainty, and validation modules visible gene by gene, the resource helps researchers avoid overinterpreting single-cohort differential-expression results.

The main finding is not that one mechanism hypothesis is definitively confirmed. The main finding is that cardiomyopathy genes differ substantially in transcript-level reproducibility and baseline-context confidence. This is exactly why a resource is needed.

The GTEx layer is particularly important. Many high-MCI genes are transcript-concordant but GTEx-low-confidence, meaning their disease-associated variability does not exceed normal left-ventricle baseline variability in the current bulk data. This does not invalidate the genes. It changes the interpretation from unqualified transcript-level target to candidate requiring orthogonal validation.

The non-significant mechanism, DCM, held-out, and GWAS analyses should be interpreted as honest evidence boundaries. In a discovery manuscript, these results would weaken the central claim. In a resource manuscript, they strengthen the credibility of the resource because they show that the framework reports discordance and uncertainty rather than hiding it.

## 5. Limitations

The current resource is limited by the number and size of public cohorts, bulk-tissue composition effects, incomplete genotype stratification, platform differences, and lack of direct wet-lab validation. GTEx provides a useful normal baseline but is not a matched disease-control cohort. DCM generalization remains limited by available datasets and gene coverage. The scRNA-seq module has been scaffolded but still requires selection and execution on a suitable public human cardiac single-cell dataset.

## 6. Conclusion

The MCI cardiomyopathy resource provides a reusable and transparent evidence-auditing layer for ClinVar-annotated cardiomyopathy genes. It distinguishes clinical annotation from transcriptomic reproducibility, normal-tissue variability, disease-context generalization, and independent validation evidence. This resource-paper framing better matches the current evidence than a high-impact discovery claim and provides a stronger foundation for future single-cell, proteomic, genotype-stratified, and large-cohort extensions.

## Data and code availability

Code and resource outputs are available in this repository. The current master resource table is located at:

`results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv`

The data dictionary is located at:

`results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_DATA_DICTIONARY_v0_2.csv`

The scRNA-seq contextualization scaffold is located at:

`scripts/scrna_contextualization/`
"""

# Save manuscript skeleton
manuscript_path = manuscript_dir / "05_RESOURCE_PAPER_MANUSCRIPT_SKELETON.md"
manuscript_path.write_text(manuscript)

# Save top tables separately
top_high_path = manuscript_dir / "TABLE_top_high_concordance_HCM_genes_v0_2.csv"
unstable_path = manuscript_dir / "TABLE_most_unstable_HCM_genes_v0_2.csv"

top_high.to_csv(top_high_path, index=False)
unstable_genes.to_csv(unstable_path, index=False)

# Metadata
summary = {
    "step": "STEP_6_RESOURCE_MANUSCRIPT_SKELETON",
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "manuscript_skeleton": str(manuscript_path.relative_to(REPO_DIR)),
    "top_high_table": str(top_high_path.relative_to(REPO_DIR)),
    "unstable_table": str(unstable_path.relative_to(REPO_DIR)),
    "n_genes": int(n_genes),
    "tier_counts": {str(k): int(v) for k, v in tier_counts.items()},
    "gtex_low_confidence_counts": {str(k): int(v) for k, v in gtex_counts.items()},
    "n_with_dcm_entry": int(n_dcm)
}

summary_path = metadata_dir / "step6_resource_manuscript_skeleton_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("STEP 6 COMPLETE.")
print("Created manuscript skeleton:", manuscript_path)
print("Created top high table:", top_high_path)
print("Created unstable table:", unstable_path)
print("Created summary:", summary_path)

print("\nSummary:")
print(json.dumps(summary, indent=2))

print("\nPreview:")
print(manuscript[:2500])

STEP 6 COMPLETE.
Created manuscript skeleton: /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/manuscript/resource_paper/05_RESOURCE_PAPER_MANUSCRIPT_SKELETON.md
Created top high table: /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/manuscript/resource_paper/TABLE_top_high_concordance_HCM_genes_v0_2.csv
Created unstable table: /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/manuscript/resource_paper/TABLE_most_unstable_HCM_genes_v0_2.csv
Created summary: /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/metadata/step6_resource_manuscript_skeleton_summary.json

Summary:
{
  "step": "STEP_6_RESOURCE_MANUSCRIPT_SKELETON",
  "timestamp_utc": "2026-07-09T21:53:53.755621+00:00",
  "manuscript_skeleton": "manuscript/resource_paper/05_RESOURCE_PAPER_MANUSCRIPT_SKELETON.md",
  "top_high_table": "manuscript/resource_paper/TABLE_top_high_concordance_HCM_genes_v0_2.csv",
  "unstable_table": "manuscript/resource_paper/TABLE_most_

In [ ]:
# =========================
# STEP 7: Create resource summary tables and figures
# =========================

from pathlib import Path
import pandas as pd
import numpy as np
import json, datetime
import matplotlib.pyplot as plt

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")
assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

resource_table_path = REPO_DIR / "results" / "resource_tables" / "MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv"
assert resource_table_path.exists(), f"Resource table not found: {resource_table_path}"

resource = pd.read_csv(resource_table_path)

summary_dir = REPO_DIR / "results" / "resource_tables"
fig_dir = REPO_DIR / "results" / "figures" / "resource_paper"
metadata_dir = REPO_DIR / "metadata"

summary_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

# -------------------------
# 1. Create tier count summary table
# -------------------------

tier_order = ["HIGH", "MODERATE", "UNSTABLE", "INSUFFICIENT_COHORT_COVERAGE"]

tier_summary = (
    resource["MCI_tier"]
    .value_counts(dropna=False)
    .rename_axis("MCI_tier")
    .reset_index(name="n_genes")
)

tier_summary["MCI_tier"] = tier_summary["MCI_tier"].astype(str)
tier_summary["tier_order"] = tier_summary["MCI_tier"].apply(
    lambda x: tier_order.index(x) if x in tier_order else 999
)
tier_summary = tier_summary.sort_values("tier_order").drop(columns=["tier_order"])

tier_summary["percent_of_resource"] = (
    100 * tier_summary["n_genes"] / tier_summary["n_genes"].sum()
).round(2)

tier_summary_path = summary_dir / "RESOURCE_SUMMARY_MCI_TIER_COUNTS_v0_2.csv"
tier_summary.to_csv(tier_summary_path, index=False)

# -------------------------
# 2. Create GTEx confidence summary table
# -------------------------

gtex_summary = (
    resource["GTEx_low_confidence_flag"]
    .value_counts(dropna=False)
    .rename_axis("GTEx_low_confidence_flag")
    .reset_index(name="n_genes")
)

gtex_summary["GTEx_low_confidence_flag"] = gtex_summary["GTEx_low_confidence_flag"].astype(str)
gtex_summary["percent_of_resource"] = (
    100 * gtex_summary["n_genes"] / gtex_summary["n_genes"].sum()
).round(2)

gtex_summary_path = summary_dir / "RESOURCE_SUMMARY_GTEx_CONFIDENCE_COUNTS_v0_2.csv"
gtex_summary.to_csv(gtex_summary_path, index=False)

# -------------------------
# 3. Create browser display table
# -------------------------

browser_cols = [
    "gene_symbol",
    "disease_group",
    "stratum",
    "MCI",
    "Adj_MCI",
    "MCI_tier",
    "bootstrap_majority_tier",
    "MCI_CI95_lower",
    "MCI_CI95_upper",
    "sigma_disease_to_GTEx_ratio",
    "GTEx_low_confidence_flag",
    "has_DCM_generalization_entry",
    "DCM_MCI_if_available",
    "DCM_tier_if_available",
    "resource_interpretation"
]

existing_browser_cols = [c for c in browser_cols if c in resource.columns]
browser_table = resource[existing_browser_cols].copy()

for col in ["MCI", "Adj_MCI", "MCI_CI95_lower", "MCI_CI95_upper", "sigma_disease_to_GTEx_ratio", "DCM_MCI_if_available"]:
    if col in browser_table.columns:
        browser_table[col] = pd.to_numeric(browser_table[col], errors="coerce").round(4)

browser_table = browser_table.sort_values(["MCI_tier", "MCI"], ascending=[True, False])

browser_table_path = summary_dir / "MCI_BROWSER_READY_DISPLAY_TABLE_v0_2.csv"
browser_table.to_csv(browser_table_path, index=False)

# -------------------------
# 4. Figure 1: MCI tier count bar plot
# -------------------------

plt.figure(figsize=(7, 5))
plot_df = tier_summary.copy()
plt.bar(plot_df["MCI_tier"], plot_df["n_genes"])
plt.xlabel("MCI tier")
plt.ylabel("Number of genes")
plt.title("MCI cardiomyopathy resource tier distribution")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

tier_fig_path = fig_dir / "RESOURCE_FIGURE_MCI_TIER_DISTRIBUTION_v0_2.png"
plt.savefig(tier_fig_path, dpi=300, bbox_inches="tight")
plt.close()

# -------------------------
# 5. Figure 2: MCI vs GTEx ratio
# -------------------------

plot_scatter = resource.copy()
plot_scatter["MCI"] = pd.to_numeric(plot_scatter["MCI"], errors="coerce")
plot_scatter["sigma_disease_to_GTEx_ratio"] = pd.to_numeric(
    plot_scatter["sigma_disease_to_GTEx_ratio"], errors="coerce"
)
plot_scatter = plot_scatter.dropna(subset=["MCI", "sigma_disease_to_GTEx_ratio"])

plt.figure(figsize=(7, 5))
plt.scatter(plot_scatter["MCI"], plot_scatter["sigma_disease_to_GTEx_ratio"])
plt.axhline(1.0, linestyle="--")
plt.axvline(0.70, linestyle="--")
plt.axvline(0.45, linestyle="--")
plt.xlabel("MCI")
plt.ylabel("sigma_disease / sigma_GTEx")
plt.title("MCI versus GTEx baseline variability ratio")
plt.tight_layout()

scatter_fig_path = fig_dir / "RESOURCE_FIGURE_MCI_vs_GTEx_RATIO_v0_2.png"
plt.savefig(scatter_fig_path, dpi=300, bbox_inches="tight")
plt.close()

# -------------------------
# 6. Figure 3: GTEx confidence counts
# -------------------------

plt.figure(figsize=(6, 5))
gtex_plot = gtex_summary.copy()
plt.bar(gtex_plot["GTEx_low_confidence_flag"], gtex_plot["n_genes"])
plt.xlabel("GTEx low-confidence flag")
plt.ylabel("Number of genes")
plt.title("GTEx baseline confidence distribution")
plt.tight_layout()

gtex_fig_path = fig_dir / "RESOURCE_FIGURE_GTEx_CONFIDENCE_DISTRIBUTION_v0_2.png"
plt.savefig(gtex_fig_path, dpi=300, bbox_inches="tight")
plt.close()

# -------------------------
# 7. Figure 4: Top 20 genes by MCI
# -------------------------

top20 = resource.sort_values("MCI", ascending=False).head(20).copy()
top20 = top20.sort_values("MCI", ascending=True)

plt.figure(figsize=(8, 7))
plt.barh(top20["gene_symbol"], top20["MCI"])
plt.axvline(0.70, linestyle="--")
plt.axvline(0.45, linestyle="--")
plt.xlabel("MCI")
plt.ylabel("Gene")
plt.title("Top 20 HCM genes by Molecular Concordance Index")
plt.tight_layout()

top20_fig_path = fig_dir / "RESOURCE_FIGURE_TOP20_HCM_MCI_GENES_v0_2.png"
plt.savefig(top20_fig_path, dpi=300, bbox_inches="tight")
plt.close()

# -------------------------
# 8. Save summary JSON
# -------------------------

summary = {
    "step": "STEP_7_RESOURCE_SUMMARY_TABLES_AND_FIGURES",
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "input_master_table": str(resource_table_path.relative_to(REPO_DIR)),
    "tables_created": [
        str(tier_summary_path.relative_to(REPO_DIR)),
        str(gtex_summary_path.relative_to(REPO_DIR)),
        str(browser_table_path.relative_to(REPO_DIR))
    ],
    "figures_created": [
        str(tier_fig_path.relative_to(REPO_DIR)),
        str(scatter_fig_path.relative_to(REPO_DIR)),
        str(gtex_fig_path.relative_to(REPO_DIR)),
        str(top20_fig_path.relative_to(REPO_DIR))
    ],
    "n_genes": int(resource["gene_symbol"].nunique()),
    "tier_counts": tier_summary.set_index("MCI_tier")["n_genes"].to_dict(),
    "gtex_counts": gtex_summary.set_index("GTEx_low_confidence_flag")["n_genes"].to_dict()
}

summary_path = metadata_dir / "step7_resource_summary_tables_and_figures_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("STEP 7 COMPLETE.")
print("\nTables created:")
for x in summary["tables_created"]:
    print(" -", x)

print("\nFigures created:")
for x in summary["figures_created"]:
    print(" -", x)

print("\nSummary:")
print(json.dumps(summary, indent=2))

print("\nBrowser table preview:")
display(browser_table.head(12))

STEP 7 COMPLETE.

Tables created:
 - results/resource_tables/RESOURCE_SUMMARY_MCI_TIER_COUNTS_v0_2.csv
 - results/resource_tables/RESOURCE_SUMMARY_GTEx_CONFIDENCE_COUNTS_v0_2.csv
 - results/resource_tables/MCI_BROWSER_READY_DISPLAY_TABLE_v0_2.csv

Figures created:
 - results/figures/resource_paper/RESOURCE_FIGURE_MCI_TIER_DISTRIBUTION_v0_2.png
 - results/figures/resource_paper/RESOURCE_FIGURE_MCI_vs_GTEx_RATIO_v0_2.png
 - results/figures/resource_paper/RESOURCE_FIGURE_GTEx_CONFIDENCE_DISTRIBUTION_v0_2.png
 - results/figures/resource_paper/RESOURCE_FIGURE_TOP20_HCM_MCI_GENES_v0_2.png

Summary:
{
  "step": "STEP_7_RESOURCE_SUMMARY_TABLES_AND_FIGURES",
  "timestamp_utc": "2026-07-09T21:57:07.992986+00:00",
  "input_master_table": "results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv",
  "tables_created": [
    "results/resource_tables/RESOURCE_SUMMARY_MCI_TIER_COUNTS_v0_2.csv",
    "results/resource_tables/RESOURCE_SUMMARY_GTEx_CONFIDENCE_COUNTS_v0_2.csv",
    "result

,gene_symbol,disease_group,stratum,MCI,Adj_MCI,MCI_tier,bootstrap_majority_tier,MCI_CI95_lower,MCI_CI95_upper,sigma_disease_to_GTEx_ratio,GTEx_low_confidence_flag,has_DCM_generalization_entry,DCM_MCI_if_available,DCM_tier_if_available,resource_interpretation
0,MYH6,HCM,NaN,0.9681,1.1844,HIGH,HIGH,0.9618,1.0,0.1675,True,False,NaN,NaN,"High cross-cohort transcript concordance, but ..."
1,TNNI3,HCM,HCM_sarcomeric,0.9617,1.0505,HIGH,HIGH,0.9537,1.0,0.0661,True,True,0.8461,HIGH,"High cross-cohort transcript concordance, but ..."
2,GLA,HCM,NaN,0.9600,1.1852,HIGH,HIGH,0.9509,1.0,0.1765,True,False,NaN,NaN,"High cross-cohort transcript concordance, but ..."
3,ACTN2,HCM,NaN,0.9348,1.0002,HIGH,HIGH,0.9177,1.0,0.0497,True,True,0.7281,HIGH,"High cross-cohort transcript concordance, but ..."
4,TMEM43,HCM,NaN,0.9295,1.2137,HIGH,HIGH,0.9043,1.0,0.2361,True,True,0.9988,HIGH,"High cross-cohort transcript concordance, but ..."
5,TINF2,HCM,NaN,0.9041,1.1995,HIGH,HIGH,0.8588,1.0,0.2542,True,False,NaN,NaN,"High cross-cohort transcript concordance, but ..."
6,KCNH2,HCM,NaN,0.8814,1.1280,HIGH,HIGH,0.8037,1.0,0.2141,True,False,NaN,NaN,"High cross-cohort transcript concordance, but ..."
7,BAG3,HCM,HCM_non_sarcomeric,0.8282,0.9766,HIGH,HIGH,0.7047,1.0,0.1322,True,True,0.6928,MODERATE,"High cross-cohort transcript concordance, but ..."
8,TRIM63,HCM,NaN,0.8198,0.8320,HIGH,HIGH,0.7448,1.0,0.0104,True,False,NaN,NaN,"High cross-cohort transcript concordance, but ..."
9,RBM20,HCM,HCM_non_sarcomeric,0.8036,1.0794,HIGH,HIGH,0.6522,1.0,0.2686,True,False,NaN,INSUFFICIENT_COVERAGE,"High cross-cohort transcript concordance, but ..."


In [ ]:
# =========================
# STEP 8: Create Shiny browser app skeleton for MCI resource
# =========================

from pathlib import Path
import json, datetime

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")
assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

shiny_dir = REPO_DIR / "shiny_app"
metadata_dir = REPO_DIR / "metadata"
shiny_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

browser_table_rel = "results/resource_tables/MCI_BROWSER_READY_DISPLAY_TABLE_v0_2.csv"
browser_table_path = REPO_DIR / browser_table_rel
assert browser_table_path.exists(), f"Browser table not found: {browser_table_path}"

# -------------------------
# 1. Create app.R
# -------------------------

app_r = r'''
# MCI Cardiomyopathy Resource Browser
# Resource version: v0.2-resource-reframe-corrected-source

library(shiny)
library(DT)
library(ggplot2)

DATA_PATH <- "../results/resource_tables/MCI_BROWSER_READY_DISPLAY_TABLE_v0_2.csv"

if (!file.exists(DATA_PATH)) {
  stop(paste("Could not find browser table at:", DATA_PATH))
}

mci <- read.csv(DATA_PATH, stringsAsFactors = FALSE, check.names = FALSE)

numeric_cols <- c(
  "MCI", "Adj_MCI", "MCI_CI95_lower", "MCI_CI95_upper",
  "sigma_disease_to_GTEx_ratio", "DCM_MCI_if_available"
)

for (col in numeric_cols) {
  if (col %in% names(mci)) {
    mci[[col]] <- suppressWarnings(as.numeric(mci[[col]]))
  }
}

tier_levels <- c("HIGH", "MODERATE", "UNSTABLE", "INSUFFICIENT_COHORT_COVERAGE")

ui <- fluidPage(
  titlePanel("MCI Cardiomyopathy Resource Browser"),

  tags$p(
    "A browser-ready resource for evaluating transcriptomic concordance of ClinVar-annotated cardiomyopathy genes across public human heart datasets."
  ),

  tags$hr(),

  sidebarLayout(
    sidebarPanel(
      width = 3,

      selectInput(
        "gene",
        "Search gene",
        choices = sort(unique(mci$gene_symbol)),
        selected = sort(unique(mci$gene_symbol))[1]
      ),

      selectInput(
        "tier",
        "Filter by MCI tier",
        choices = c("ALL", sort(unique(mci$MCI_tier))),
        selected = "ALL"
      ),

      selectInput(
        "gtex_flag",
        "Filter by GTEx low-confidence flag",
        choices = c("ALL", "TRUE", "FALSE"),
        selected = "ALL"
      ),

      checkboxInput(
        "only_dcm",
        "Show only genes with DCM generalization entry",
        value = FALSE
      ),

      tags$hr(),

      downloadButton("download_filtered", "Download filtered table"),

      tags$hr(),

      tags$p(
        tags$b("Interpretation boundary:")
      ),
      tags$p(
        "MCI is an evidence-auditing resource. It does not prove causality, clinical actionability, or therapeutic validity."
      )
    ),

    mainPanel(
      width = 9,

      tabsetPanel(
        tabPanel(
          "Gene view",
          br(),
          h3(textOutput("gene_title")),
          tableOutput("gene_summary"),
          br(),
          h4("Resource interpretation"),
          verbatimTextOutput("gene_interpretation"),
          br(),
          h4("MCI and GTEx context"),
          plotOutput("gene_plot", height = "300px")
        ),

        tabPanel(
          "Resource table",
          br(),
          DTOutput("resource_table")
        ),

        tabPanel(
          "Tier distribution",
          br(),
          plotOutput("tier_plot", height = "400px")
        ),

        tabPanel(
          "MCI vs GTEx ratio",
          br(),
          plotOutput("mci_gtex_plot", height = "450px")
        ),

        tabPanel(
          "About",
          br(),
          h3("About this resource"),
          tags$p(
            "The Molecular Concordance Index combines direction agreement, effect-size consistency, and statistical reproducibility across disease cohorts."
          ),
          tags$p(
            "GTEx baseline benchmarking compares disease-associated variability against normal left-ventricle expression variability."
          ),
          tags$p(
            "A GTEx-low-confidence flag means disease-associated variability does not exceed GTEx baseline variability in the current bulk data. It does not mean the gene is biologically irrelevant."
          ),
          tags$p(
            "This browser is designed for database/resource manuscript framing and gene-level evidence review."
          )
        )
      )
    )
  )
)

server <- function(input, output, session) {

  filtered_data <- reactive({
    df <- mci

    if (!is.null(input$tier) && input$tier != "ALL") {
      df <- df[df$MCI_tier == input$tier, ]
    }

    if (!is.null(input$gtex_flag) && input$gtex_flag != "ALL") {
      target_flag <- input$gtex_flag == "TRUE"
      df <- df[toupper(as.character(df$GTEx_low_confidence_flag)) == as.character(target_flag), ]
    }

    if (isTRUE(input$only_dcm)) {
      df <- df[toupper(as.character(df$has_DCM_generalization_entry)) == "TRUE", ]
    }

    df
  })

  selected_gene_data <- reactive({
    df <- mci[mci$gene_symbol == input$gene, ]
    if (nrow(df) == 0) {
      return(NULL)
    }
    df[1, ]
  })

  output$gene_title <- renderText({
    paste("Gene:", input$gene)
  })

  output$gene_summary <- renderTable({
    row <- selected_gene_data()
    if (is.null(row)) return(NULL)

    keep <- c(
      "gene_symbol",
      "disease_group",
      "stratum",
      "MCI",
      "Adj_MCI",
      "MCI_tier",
      "bootstrap_majority_tier",
      "MCI_CI95_lower",
      "MCI_CI95_upper",
      "sigma_disease_to_GTEx_ratio",
      "GTEx_low_confidence_flag",
      "has_DCM_generalization_entry",
      "DCM_MCI_if_available",
      "DCM_tier_if_available"
    )

    keep <- keep[keep %in% names(row)]
    out <- data.frame(
      field = keep,
      value = as.character(unlist(row[keep])),
      stringsAsFactors = FALSE
    )
    out
  }, striped = TRUE, bordered = TRUE)

  output$gene_interpretation <- renderText({
    row <- selected_gene_data()
    if (is.null(row)) return("No gene selected.")
    if ("resource_interpretation" %in% names(row)) {
      return(row$resource_interpretation)
    }
    "No interpretation field available."
  })

  output$gene_plot <- renderPlot({
    row <- selected_gene_data()
    if (is.null(row)) return(NULL)

    vals <- data.frame(
      metric = c("MCI", "Adj_MCI", "DCM MCI"),
      value = c(row$MCI, row$Adj_MCI, row$DCM_MCI_if_available)
    )

    vals <- vals[!is.na(vals$value), ]

    ggplot(vals, aes(x = metric, y = value)) +
      geom_col() +
      geom_hline(yintercept = 0.70, linetype = "dashed") +
      geom_hline(yintercept = 0.45, linetype = "dashed") +
      ylim(0, max(1.2, max(vals$value, na.rm = TRUE))) +
      labs(
        x = "",
        y = "Score",
        title = paste("MCI profile for", row$gene_symbol)
      ) +
      theme_minimal(base_size = 13)
  })

  output$resource_table <- renderDT({
    datatable(
      filtered_data(),
      options = list(
        pageLength = 15,
        scrollX = TRUE,
        autoWidth = TRUE
      ),
      rownames = FALSE,
      filter = "top"
    )
  })

  output$tier_plot <- renderPlot({
    df <- filtered_data()
    counts <- as.data.frame(table(df$MCI_tier), stringsAsFactors = FALSE)
    names(counts) <- c("MCI_tier", "n_genes")

    ggplot(counts, aes(x = MCI_tier, y = n_genes)) +
      geom_col() +
      labs(
        x = "MCI tier",
        y = "Number of genes",
        title = "MCI tier distribution"
      ) +
      theme_minimal(base_size = 13) +
      theme(axis.text.x = element_text(angle = 30, hjust = 1))
  })

  output$mci_gtex_plot <- renderPlot({
    df <- filtered_data()
    df <- df[!is.na(df$MCI) & !is.na(df$sigma_disease_to_GTEx_ratio), ]

    ggplot(df, aes(x = MCI, y = sigma_disease_to_GTEx_ratio)) +
      geom_point() +
      geom_hline(yintercept = 1.0, linetype = "dashed") +
      geom_vline(xintercept = 0.70, linetype = "dashed") +
      geom_vline(xintercept = 0.45, linetype = "dashed") +
      labs(
        x = "MCI",
        y = "sigma_disease / sigma_GTEx",
        title = "MCI versus GTEx baseline variability ratio"
      ) +
      theme_minimal(base_size = 13)
  })

  output$download_filtered <- downloadHandler(
    filename = function() {
      paste0("MCI_filtered_resource_table_", Sys.Date(), ".csv")
    },
    content = function(file) {
      write.csv(filtered_data(), file, row.names = FALSE)
    }
  )
}

shinyApp(ui = ui, server = server)
'''

app_path = shiny_dir / "app.R"
app_path.write_text(app_r)

# -------------------------
# 2. Create Shiny README
# -------------------------

shiny_readme = f'''
# MCI Cardiomyopathy Resource Browser

This Shiny app provides an interactive browser for the MCI cardiomyopathy resource.

## Input table

The app reads:

`../{browser_table_rel}`

## Main features

- Gene-level search
- MCI, adjusted MCI, tier, and bootstrap interval display
- GTEx low-confidence flag display
- DCM generalization fields where available
- Full searchable and downloadable resource table
- Tier distribution plot
- MCI versus GTEx variability-ratio plot

## Local run

From the `shiny_app` folder, run:

```r
shiny::runApp()

SyntaxError: incomplete input (1113299205.py, line 326)

In [ ]:
# =========================
# STEP 8 FIXED: Create Shiny browser app skeleton safely
# =========================

from pathlib import Path
import json, datetime

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")
assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

shiny_dir = REPO_DIR / "shiny_app"
metadata_dir = REPO_DIR / "metadata"
shiny_dir.mkdir(parents=True, exist_ok=True)
metadata_dir.mkdir(parents=True, exist_ok=True)

browser_table_rel = "results/resource_tables/MCI_BROWSER_READY_DISPLAY_TABLE_v0_2.csv"
browser_table_path = REPO_DIR / browser_table_rel
assert browser_table_path.exists(), f"Browser table not found: {browser_table_path}"

# -------------------------
# 1. Write app.R safely
# -------------------------

app_lines = [
"# MCI Cardiomyopathy Resource Browser",
"# Resource version: v0.2-resource-reframe-corrected-source",
"",
"library(shiny)",
"library(DT)",
"library(ggplot2)",
"",
'DATA_PATH <- "../results/resource_tables/MCI_BROWSER_READY_DISPLAY_TABLE_v0_2.csv"',
"",
"if (!file.exists(DATA_PATH)) {",
'  stop(paste("Could not find browser table at:", DATA_PATH))',
"}",
"",
"mci <- read.csv(DATA_PATH, stringsAsFactors = FALSE, check.names = FALSE)",
"",
'numeric_cols <- c("MCI", "Adj_MCI", "MCI_CI95_lower", "MCI_CI95_upper", "sigma_disease_to_GTEx_ratio", "DCM_MCI_if_available")',
"",
"for (col in numeric_cols) {",
"  if (col %in% names(mci)) {",
"    mci[[col]] <- suppressWarnings(as.numeric(mci[[col]]))",
"  }",
"}",
"",
'ui <- fluidPage(',
'  titlePanel("MCI Cardiomyopathy Resource Browser"),',
'  tags$p("A browser-ready resource for evaluating transcriptomic concordance of ClinVar-annotated cardiomyopathy genes across public human heart datasets."),',
'  tags$hr(),',
'  sidebarLayout(',
'    sidebarPanel(',
'      width = 3,',
'      selectInput("gene", "Search gene", choices = sort(unique(mci$gene_symbol)), selected = sort(unique(mci$gene_symbol))[1]),',
'      selectInput("tier", "Filter by MCI tier", choices = c("ALL", sort(unique(mci$MCI_tier))), selected = "ALL"),',
'      selectInput("gtex_flag", "Filter by GTEx low-confidence flag", choices = c("ALL", "TRUE", "FALSE"), selected = "ALL"),',
'      checkboxInput("only_dcm", "Show only genes with DCM generalization entry", value = FALSE),',
'      tags$hr(),',
'      downloadButton("download_filtered", "Download filtered table"),',
'      tags$hr(),',
'      tags$p(tags$b("Interpretation boundary:")),',
'      tags$p("MCI is an evidence-auditing resource. It does not prove causality, clinical actionability, or therapeutic validity.")',
'    ),',
'    mainPanel(',
'      width = 9,',
'      tabsetPanel(',
'        tabPanel("Gene view", br(), h3(textOutput("gene_title")), tableOutput("gene_summary"), br(), h4("Resource interpretation"), verbatimTextOutput("gene_interpretation"), br(), h4("MCI and GTEx context"), plotOutput("gene_plot", height = "300px")),',
'        tabPanel("Resource table", br(), DTOutput("resource_table")),',
'        tabPanel("Tier distribution", br(), plotOutput("tier_plot", height = "400px")),',
'        tabPanel("MCI vs GTEx ratio", br(), plotOutput("mci_gtex_plot", height = "450px")),',
'        tabPanel("About", br(), h3("About this resource"),',
'                 tags$p("The Molecular Concordance Index combines direction agreement, effect-size consistency, and statistical reproducibility across disease cohorts."),',
'                 tags$p("GTEx baseline benchmarking compares disease-associated variability against normal left-ventricle expression variability."),',
'                 tags$p("A GTEx-low-confidence flag means disease-associated variability does not exceed GTEx baseline variability in the current bulk data. It does not mean the gene is biologically irrelevant."),',
'                 tags$p("This browser is designed for database/resource manuscript framing and gene-level evidence review."))',
'      )',
'    )',
'  )',
')',
"",
"server <- function(input, output, session) {",
"",
"  filtered_data <- reactive({",
"    df <- mci",
'    if (!is.null(input$tier) && input$tier != "ALL") { df <- df[df$MCI_tier == input$tier, ] }',
'    if (!is.null(input$gtex_flag) && input$gtex_flag != "ALL") {',
'      target_flag <- ifelse(input$gtex_flag == "TRUE", "TRUE", "FALSE")',
'      df <- df[toupper(as.character(df$GTEx_low_confidence_flag)) == target_flag, ]',
"    }",
"    if (isTRUE(input$only_dcm)) {",
'      df <- df[toupper(as.character(df$has_DCM_generalization_entry)) == "TRUE", ]',
"    }",
"    df",
"  })",
"",
"  selected_gene_data <- reactive({",
"    df <- mci[mci$gene_symbol == input$gene, ]",
"    if (nrow(df) == 0) return(NULL)",
"    df[1, ]",
"  })",
"",
'  output$gene_title <- renderText({ paste("Gene:", input$gene) })',
"",
"  output$gene_summary <- renderTable({",
"    row <- selected_gene_data()",
"    if (is.null(row)) return(NULL)",
'    keep <- c("gene_symbol", "disease_group", "stratum", "MCI", "Adj_MCI", "MCI_tier", "bootstrap_majority_tier", "MCI_CI95_lower", "MCI_CI95_upper", "sigma_disease_to_GTEx_ratio", "GTEx_low_confidence_flag", "has_DCM_generalization_entry", "DCM_MCI_if_available", "DCM_tier_if_available")',
"    keep <- keep[keep %in% names(row)]",
"    out <- data.frame(field = keep, value = as.character(unlist(row[keep])), stringsAsFactors = FALSE)",
"    out",
"  }, striped = TRUE, bordered = TRUE)",
"",
"  output$gene_interpretation <- renderText({",
"    row <- selected_gene_data()",
'    if (is.null(row)) return("No gene selected.")',
'    if ("resource_interpretation" %in% names(row)) return(row$resource_interpretation)',
'    "No interpretation field available."',
"  })",
"",
"  output$gene_plot <- renderPlot({",
"    row <- selected_gene_data()",
"    if (is.null(row)) return(NULL)",
'    vals <- data.frame(metric = c("MCI", "Adj_MCI", "DCM MCI"), value = c(row$MCI, row$Adj_MCI, row$DCM_MCI_if_available))',
"    vals <- vals[!is.na(vals$value), ]",
"    ggplot(vals, aes(x = metric, y = value)) +",
"      geom_col() +",
"      geom_hline(yintercept = 0.70, linetype = 'dashed') +",
"      geom_hline(yintercept = 0.45, linetype = 'dashed') +",
"      ylim(0, max(1.2, max(vals$value, na.rm = TRUE))) +",
'      labs(x = "", y = "Score", title = paste("MCI profile for", row$gene_symbol)) +',
"      theme_minimal(base_size = 13)",
"  })",
"",
"  output$resource_table <- renderDT({",
"    datatable(filtered_data(), options = list(pageLength = 15, scrollX = TRUE, autoWidth = TRUE), rownames = FALSE, filter = 'top')",
"  })",
"",
"  output$tier_plot <- renderPlot({",
"    df <- filtered_data()",
"    counts <- as.data.frame(table(df$MCI_tier), stringsAsFactors = FALSE)",
'    names(counts) <- c("MCI_tier", "n_genes")',
"    ggplot(counts, aes(x = MCI_tier, y = n_genes)) +",
"      geom_col() +",
'      labs(x = "MCI tier", y = "Number of genes", title = "MCI tier distribution") +',
"      theme_minimal(base_size = 13) +",
"      theme(axis.text.x = element_text(angle = 30, hjust = 1))",
"  })",
"",
"  output$mci_gtex_plot <- renderPlot({",
"    df <- filtered_data()",
"    df <- df[!is.na(df$MCI) & !is.na(df$sigma_disease_to_GTEx_ratio), ]",
"    ggplot(df, aes(x = MCI, y = sigma_disease_to_GTEx_ratio)) +",
"      geom_point() +",
"      geom_hline(yintercept = 1.0, linetype = 'dashed') +",
"      geom_vline(xintercept = 0.70, linetype = 'dashed') +",
"      geom_vline(xintercept = 0.45, linetype = 'dashed') +",
'      labs(x = "MCI", y = "sigma_disease / sigma_GTEx", title = "MCI versus GTEx baseline variability ratio") +',
"      theme_minimal(base_size = 13)",
"  })",
"",
"  output$download_filtered <- downloadHandler(",
"    filename = function() { paste0('MCI_filtered_resource_table_', Sys.Date(), '.csv') },",
"    content = function(file) { write.csv(filtered_data(), file, row.names = FALSE) }",
"  )",
"}",
"",
"shinyApp(ui = ui, server = server)",
""
]

app_path = shiny_dir / "app.R"
app_path.write_text("\n".join(app_lines))

# -------------------------
# 2. Write README safely
# -------------------------

readme_lines = [
"# MCI Cardiomyopathy Resource Browser",
"",
"This Shiny app provides an interactive browser for the MCI cardiomyopathy resource.",
"",
"## Input table",
"",
"../results/resource_tables/MCI_BROWSER_READY_DISPLAY_TABLE_v0_2.csv",
"",
"## Main features",
"",
"- Gene-level search",
"- MCI, adjusted MCI, tier, and bootstrap interval display",
"- GTEx low-confidence flag display",
"- DCM generalization fields where available",
"- Full searchable and downloadable resource table",
"- Tier distribution plot",
"- MCI versus GTEx variability-ratio plot",
"",
"## Local run",
"",
"From the repository root, run in R:",
"",
"shiny::runApp('shiny_app')",
"",
"Required R packages:",
"",
"install.packages(c('shiny', 'DT', 'ggplot2'))",
"",
"## Interpretation boundary",
"",
"This app is an evidence-auditing browser. It does not prove clinical actionability, therapeutic validity, or disease causality.",
""
]

(shiny_dir / "README.md").write_text("\n".join(readme_lines))

# -------------------------
# 3. Write deployment notes safely
# -------------------------

deploy_lines = [
"# Shiny Deployment Notes",
"",
"Recommended deployment target:",
"",
"- shinyapps.io for manuscript review",
"- persistent URL before manuscript submission",
"",
"Before deployment:",
"",
"1. Confirm results/resource_tables/MCI_BROWSER_READY_DISPLAY_TABLE_v0_2.csv exists.",
"2. Confirm shiny_app/app.R loads the table correctly.",
"3. Run locally with shiny::runApp('shiny_app').",
"4. Deploy using rsconnect.",
"",
"Example R commands:",
"",
"install.packages('rsconnect')",
"library(rsconnect)",
"",
"rsconnect::setAccountInfo(",
"  name = 'YOUR_SHINYAPPS_NAME',",
"  token = 'YOUR_TOKEN',",
"  secret = 'YOUR_SECRET'",
")",
"",
"rsconnect::deployApp('shiny_app')",
"",
"Do not commit tokens or secrets to GitHub.",
""
]

(shiny_dir / "DEPLOYMENT_NOTES.md").write_text("\n".join(deploy_lines))

# -------------------------
# 4. Save metadata
# -------------------------

summary = {
    "step": "STEP_8_FIXED_SHINY_BROWSER_APP_SKELETON",
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "browser_table": browser_table_rel,
    "files_created": [
        "shiny_app/app.R",
        "shiny_app/README.md",
        "shiny_app/DEPLOYMENT_NOTES.md"
    ],
    "status": "Shiny browser skeleton created from v0.2 browser-ready resource table."
}

summary_path = metadata_dir / "step8_shiny_browser_app_skeleton_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("STEP 8 FIXED COMPLETE.")
print("Created:")
for f in summary["files_created"]:
    print(" -", f)
print("\nSummary:")
print(json.dumps(summary, indent=2))

print("\nFirst 40 lines of app.R:")
print("\n".join(app_lines[:40]))

STEP 8 FIXED COMPLETE.
Created:
 - shiny_app/app.R
 - shiny_app/README.md
 - shiny_app/DEPLOYMENT_NOTES.md

Summary:
{
  "step": "STEP_8_FIXED_SHINY_BROWSER_APP_SKELETON",
  "timestamp_utc": "2026-07-09T22:15:23.687142+00:00",
  "browser_table": "results/resource_tables/MCI_BROWSER_READY_DISPLAY_TABLE_v0_2.csv",
  "files_created": [
    "shiny_app/app.R",
    "shiny_app/README.md",
    "shiny_app/DEPLOYMENT_NOTES.md"
  ],
  "status": "Shiny browser skeleton created from v0.2 browser-ready resource table."
}

First 40 lines of app.R:
# MCI Cardiomyopathy Resource Browser
# Resource version: v0.2-resource-reframe-corrected-source

library(shiny)
library(DT)
library(ggplot2)

DATA_PATH <- "../results/resource_tables/MCI_BROWSER_READY_DISPLAY_TABLE_v0_2.csv"

if (!file.exists(DATA_PATH)) {
  stop(paste("Could not find browser table at:", DATA_PATH))
}

mci <- read.csv(DATA_PATH, stringsAsFactors = FALSE, check.names = FALSE)

numeric_cols <- c("MCI", "Adj_MCI", "MCI_CI95_lower", "MCI_CI95_

In [ ]:
# =========================
# STEP 9: Create GitHub-safe push manifest before push
# =========================

from pathlib import Path
import pandas as pd
import json, datetime, subprocess, os

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")
assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

metadata_dir = REPO_DIR / "metadata"
qc_dir = REPO_DIR / "results" / "quality_control" / "resource_tables"
metadata_dir.mkdir(parents=True, exist_ok=True)
qc_dir.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, cwd=REPO_DIR, check=False):
    print("\n$", cmd)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")
    return result

# -------------------------
# 1. Update .gitignore safely
# -------------------------

gitignore_path = REPO_DIR / ".gitignore"

new_gitignore_rules = [
    "",
    "# Resource reframe safety rules",
    ".env",
    "*.token",
    "*token*",
    "*secret*",
    ".ipynb_checkpoints/",
    "__pycache__/",
    "*.pyc",
    "",
    "# Large/raw biological data should not be committed",
    "data/raw/",
    "data/external/gtex_v8/",
    "*.gct",
    "*.gct.gz",
    "*.vcf",
    "*.vcf.gz",
    "*.tar",
    "*.tar.gz",
    "*.zip",
    "",
    "# Large expression matrices",
    "*expression_matrix*.csv",
    "*gene_symbol_matrix*.csv",
    "*raw_CODE*.csv",
    "",
    "# Local runtime files",
    ".Rhistory",
    ".RData",
    ".DS_Store",
]

existing = gitignore_path.read_text() if gitignore_path.exists() else ""
to_add = []
for rule in new_gitignore_rules:
    if rule not in existing:
        to_add.append(rule)

if to_add:
    with gitignore_path.open("a") as f:
        f.write("\n".join(to_add))
        f.write("\n")

print("Updated .gitignore:", gitignore_path)

# -------------------------
# 2. Define intended GitHub-safe files
# -------------------------

safe_patterns = [
    "README_RESOURCE.md",
    ".gitignore",

    "manuscript/resource_paper/*.md",
    "manuscript/resource_paper/*.csv",

    "docs/resource_framing/*.md",

    "results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_MASTER_TABLE_v0_2.csv",
    "results/resource_tables/MCI_CARDIOMYOPATHY_RESOURCE_DATA_DICTIONARY_v0_2.csv",
    "results/resource_tables/MCI_scRNA_contextualization_target_gene_list_v0_1.csv",
    "results/resource_tables/RESOURCE_SUMMARY_MCI_TIER_COUNTS_v0_2.csv",
    "results/resource_tables/RESOURCE_SUMMARY_GTEx_CONFIDENCE_COUNTS_v0_2.csv",
    "results/resource_tables/MCI_BROWSER_READY_DISPLAY_TABLE_v0_2.csv",

    "results/figures/resource_paper/*.png",

    "scripts/scrna_contextualization/*.py",
    "scripts/scrna_contextualization/*.md",

    "shiny_app/app.R",
    "shiny_app/README.md",
    "shiny_app/DEPLOYMENT_NOTES.md",

    "metadata/step2_resource_framing_metadata.json",
    "metadata/step4_resource_readme_documentation_summary.json",
    "metadata/step5_scrna_contextualization_scaffold_summary.json",
    "metadata/step6_resource_manuscript_skeleton_summary.json",
    "metadata/step7_resource_summary_tables_and_figures_summary.json",
    "metadata/step8_shiny_browser_app_skeleton_summary.json",
]

safe_files = []
for pattern in safe_patterns:
    safe_files.extend(REPO_DIR.glob(pattern))

safe_files = sorted(set([p for p in safe_files if p.exists() and p.is_file()]))

# -------------------------
# 3. Safety checks
# -------------------------

forbidden_fragments = [
    "data/external/gtex_v8",
    "data/raw",
    ".env",
    "token",
    "secret",
    ".ipynb_checkpoints",
    "__pycache__",
]

manifest_rows = []
for p in safe_files:
    rel = str(p.relative_to(REPO_DIR))
    size_mb = p.stat().st_size / (1024 * 1024)
    forbidden_hit = any(frag.lower() in rel.lower() for frag in forbidden_fragments)

    # Most files should be small. Figures can be a bit larger, but flag anything > 20 MB.
    size_safe = size_mb <= 20

    github_safe = (not forbidden_hit) and size_safe

    manifest_rows.append({
        "relative_path": rel,
        "size_mb": round(size_mb, 6),
        "github_safe": github_safe,
        "forbidden_fragment_hit": forbidden_hit,
        "size_safe_under_20mb": size_safe,
    })

manifest = pd.DataFrame(manifest_rows)

manifest_path = qc_dir / "STEP9_GITHUB_SAFE_RESOURCE_PUSH_MANIFEST.csv"
manifest.to_csv(manifest_path, index=False)

# Include manifest itself in staging
safe_files.append(manifest_path)
safe_files = sorted(set(safe_files))

# -------------------------
# 4. Save JSON summary
# -------------------------

summary = {
    "step": "STEP_9_GITHUB_SAFE_PUSH_MANIFEST",
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "n_safe_files_detected": int(len(safe_files)),
    "manifest_path": str(manifest_path.relative_to(REPO_DIR)),
    "all_manifest_entries_safe": bool(manifest["github_safe"].all()) if len(manifest) else False,
    "unsafe_entries": manifest.loc[~manifest["github_safe"], "relative_path"].tolist() if len(manifest) else [],
}

summary_path = metadata_dir / "step9_github_safe_push_manifest_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

safe_files.append(summary_path)
safe_files = sorted(set(safe_files))

# -------------------------
# 5. Show output
# -------------------------

print("\nSTEP 9 COMPLETE.")
print("Manifest saved:", manifest_path)
print("Summary saved:", summary_path)

print("\nManifest preview:")
display(manifest)

print("\nUnsafe entries:")
if len(summary["unsafe_entries"]) == 0:
    print("None.")
else:
    for x in summary["unsafe_entries"]:
        print(" -", x)

print("\nGit status before staging:")
run_cmd("git status --short")

print("\nFiles intended for staging:")
for p in safe_files:
    print(" -", p.relative_to(REPO_DIR))

Updated .gitignore: /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/.gitignore

STEP 9 COMPLETE.
Manifest saved: /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/results/quality_control/resource_tables/STEP9_GITHUB_SAFE_RESOURCE_PUSH_MANIFEST.csv
Summary saved: /content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance/metadata/step9_github_safe_push_manifest_summary.json

Manifest preview:


,relative_path,size_mb,github_safe,forbidden_fragment_hit,size_safe_under_20mb
0,.gitignore,0.006901,True,False,True
1,README_RESOURCE.md,0.004229,True,False,True
2,docs/resource_framing/CLAIM_BOUNDARY_STATEMENT.md,0.001133,True,False,True
3,docs/resource_framing/NON_SIGNIFICANT_RESULTS_...,0.001799,True,False,True
4,docs/resource_framing/RESOURCE_TABLE_USAGE_GUI...,0.002161,True,False,True
5,manuscript/resource_paper/00_RESOURCE_PAPER_PO...,0.002743,True,False,True
6,manuscript/resource_paper/01_REVISED_RESOURCE_...,0.002532,True,False,True
7,manuscript/resource_paper/02_REVISED_CONTRIBUT...,0.001876,True,False,True
8,manuscript/resource_paper/03_RESOURCE_MANUSCRI...,0.003248,True,False,True
9,manuscript/resource_paper/04_OPTIONAL_scRNA_CO...,0.001254,True,False,True



Unsafe entries:
None.

Git status before staging:

$ git status --short
 M .gitignore
 M results/quality_control/gtex_baseline_adjustment/TASK5_GTEx_v8_Left_Ventricle_sample_ids.csv
 M results/quality_control/step9_validation/TASK9_primary_MCI_gene_set_used_for_validation.csv
?? README_RESOURCE.md
?? docs/
?? manuscript/
?? metadata/
?? results/figures/resource_paper/
?? results/quality_control/resource_tables/
?? results/resource_tables/
?? scripts/scrna_contextualization/
?? shiny_app/DEPLOYMENT_NOTES.md
?? shiny_app/README.md
?? shiny_app/app.R


Files intended for staging:
 - .gitignore
 - README_RESOURCE.md
 - docs/resource_framing/CLAIM_BOUNDARY_STATEMENT.md
 - docs/resource_framing/NON_SIGNIFICANT_RESULTS_FRAMING.md
 - docs/resource_framing/RESOURCE_TABLE_USAGE_GUIDE.md
 - manuscript/resource_paper/00_RESOURCE_PAPER_POSITIONING.md
 - manuscript/resource_paper/01_REVISED_RESOURCE_ABSTRACT.md
 - manuscript/resource_paper/02_REVISED_CONTRIBUTIONS.md
 - manuscript/resource_paper/03

In [ ]:
# =========================
# STEP 10: Stage and commit GitHub-safe resource files only
# =========================

from pathlib import Path
import pandas as pd
import subprocess, json, datetime, shlex

REPO_DIR = Path("/content/drive/MyDrive/MCI_Project/mci-cardiomyopathy-concordance")
assert REPO_DIR.exists(), f"Repo not found: {REPO_DIR}"

manifest_path = REPO_DIR / "results" / "quality_control" / "resource_tables" / "STEP9_GITHUB_SAFE_RESOURCE_PUSH_MANIFEST.csv"
assert manifest_path.exists(), f"Manifest not found: {manifest_path}"

metadata_dir = REPO_DIR / "metadata"
metadata_dir.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, cwd=REPO_DIR, check=False):
    print("\n$", cmd)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")
    return result

# -------------------------
# 1. Load safe manifest
# -------------------------

manifest = pd.read_csv(manifest_path)

unsafe = manifest[manifest["github_safe"] != True]
if len(unsafe) > 0:
    print("Unsafe files found. Stop before staging.")
    display(unsafe)
    raise RuntimeError("Manifest contains unsafe files.")

safe_paths = manifest["relative_path"].dropna().astype(str).tolist()

# Add Step 9 manifest and summary explicitly
extra_paths = [
    "results/quality_control/resource_tables/STEP9_GITHUB_SAFE_RESOURCE_PUSH_MANIFEST.csv",
    "metadata/step9_github_safe_push_manifest_summary.json"
]

for p in extra_paths:
    if p not in safe_paths:
        safe_paths.append(p)

# Verify all intended files exist
missing = []
for rel in safe_paths:
    if not (REPO_DIR / rel).exists():
        missing.append(rel)

if missing:
    print("Missing files:")
    for m in missing:
        print(" -", m)
    raise FileNotFoundError("Some files from safe manifest are missing.")

print(f"Files to stage: {len(safe_paths)}")
for rel in safe_paths:
    print(" -", rel)

# -------------------------
# 2. Confirm current branch
# -------------------------

run_cmd("git branch --show-current")
run_cmd("git status --short")

# -------------------------
# 3. Stage only safe files
# -------------------------

# First unstage everything, just in case
run_cmd("git reset", check=False)

# Stage each file safely
for rel in safe_paths:
    run_cmd("git add -- " + shlex.quote(rel), check=True)

print("\nStaged status:")
run_cmd("git status --short")

# -------------------------
# 4. Commit staged files
# -------------------------

commit_message = "Reframe MCI as cardiomyopathy resource paper with browser-ready outputs"

commit_result = run_cmd(f'git commit -m "{commit_message}"', check=False)

# If nothing to commit, do not fail
nothing_to_commit = "nothing to commit" in commit_result.stdout.lower()

# -------------------------
# 5. Save commit summary
# -------------------------

latest_commit = run_cmd("git log -1 --oneline", check=False).stdout.strip()
status_after = run_cmd("git status --short", check=False).stdout.strip()

summary = {
    "step": "STEP_10_STAGE_AND_COMMIT_RESOURCE_FILES",
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "n_files_staged_from_manifest": len(safe_paths),
    "commit_message": commit_message,
    "latest_commit": latest_commit,
    "nothing_to_commit": nothing_to_commit,
    "status_after_commit": status_after,
    "note": "Only manifest-approved GitHub-safe resource-paper files were staged. Unrelated modified validation/GTEx files were intentionally not staged."
}

summary_path = metadata_dir / "step10_stage_and_commit_resource_files_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("\nSTEP 10 COMPLETE.")
print("Summary saved:", summary_path)
print(json.dumps(summary, indent=2))

print("\nIMPORTANT:")
print("If git status still shows modified TASK5 or TASK9 files, that is okay because we intentionally did not stage them.")
print("Next step will push the committed branch to GitHub.")

Files to stage: 37
 - .gitignore
 - README_RESOURCE.md
 - docs/resource_framing/CLAIM_BOUNDARY_STATEMENT.md
 - docs/resource_framing/NON_SIGNIFICANT_RESULTS_FRAMING.md
 - docs/resource_framing/RESOURCE_TABLE_USAGE_GUIDE.md
 - manuscript/resource_paper/00_RESOURCE_PAPER_POSITIONING.md
 - manuscript/resource_paper/01_REVISED_RESOURCE_ABSTRACT.md
 - manuscript/resource_paper/02_REVISED_CONTRIBUTIONS.md
 - manuscript/resource_paper/03_RESOURCE_MANUSCRIPT_OUTLINE.md
 - manuscript/resource_paper/04_OPTIONAL_scRNA_CONTEXTUALIZATION_MODULE_TEXT.md
 - manuscript/resource_paper/05_RESOURCE_PAPER_MANUSCRIPT_SKELETON.md
 - manuscript/resource_paper/README.md
 - manuscript/resource_paper/TABLE_most_unstable_HCM_genes_v0_2.csv
 - manuscript/resource_paper/TABLE_top_high_concordance_HCM_genes_v0_2.csv
 - metadata/step2_resource_framing_metadata.json
 - metadata/step4_resource_readme_documentation_summary.json
 - metadata/step5_scrna_contextualization_scaffold_summary.json
 - metadata/step6_resource_m